# IV-3 音と部屋の音響学（ノートブック版）

教科書の IV-3 の 6 節のプログラムを、1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。全部で 1〜2 分です）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 共通ライブラリ `common/room.py` を pyroomacoustics と照合する | A | `room_check.py` |
| 2 | 音速・音圧・FDTD・距離減衰・A 特性 | A | `sound_basics.py` |
| 3 | インパルス応答の中身（反射の密度・DRR・直接音／初期反射／残響）を見て、聞く | B | `rir_anatomy.py` |
| 4 | サビン・アイリングと鏡像法 | C | `sabine_ism.py` |
| 5 | シュレーダー法と雑音の床、T60 の不確かさ | D | `noise_floor.py` |
| 6 | ESS・TSP・非同期の録音のシミュレーション | E | `ess_sim.py` |
| 7 | 合成の録音でインパルス応答を測る | F（試し） | `measure_ir.py --simulate` |
| 8 | 場所を単位にした T60 の集計 | G（試し） | `t60_stats.py --simulate` |
| 9 | 部屋のモードと指向性 | H | `modes_directivity.py` |
| 10 | 確認問題の答え合わせ | I | `quiz_check.py` |

- 次のセルたちが、共通ライブラリ（`common/` の `__init__.py`・`stft.py`・`audio_io.py`・`plotting.py`（IV-2）、`filters.py`（IV-1）、`array.py`（IV-5）、`room.py`（この章））と IV-2 の `iv2/sounds.py` を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の各章の 6 節と同じなので、ほかのファイルは要りません。
- 各節のコードは 6 節のプログラムそのものです。変えたのは、ファイルの場所（`__file__` の代わりにノートブックの隣を使う）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、コマンドの引数を `sys.argv` で渡すこと、出力をいったん `cap` に受け取って次のセルで表示し、`ok()` で教科書の手順の「確かめること」を確かめることだけです。
- **入れていないもの**：部屋に音を出して測る実機の手順（F の `measure_ir.py --mac`、G の `--write-sweep`・`--analyze` と、スマホで録った CSV の集計 `t60_stats.py --csv`）は、Colab ではスピーカとマイクが使えないので入れていません。Mac のターミナルで実行してください。室温（手順 A の BME280）も、ここでは 20 ℃ にしてあります。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。保存した wav をそのままの大きさで鳴らすため、`IPython.display.Audio(..., normalize=False)` を使います。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を小さくしてください。
- 話し声（3 節）は librosa の例の音源 `libri1` を使います（初回はダウンロードします。できないときは合成の声に切り替わります）。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、pyroomacoustics・scipy の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・statsmodels・matplotlib・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"), ("pyroomacoustics", "pyroomacoustics"), ("statsmodels", "statsmodels")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, librosa, pyroomacoustics
print("numpy", numpy.__version__, "scipy", scipy.__version__, "librosa", librosa.__version__, "pyroomacoustics", pyroomacoustics.__version__)
import os
for d in ["common", "iv2", "out", "data"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out

In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x

In [ ]:
%%writefile common/plotting.py
"""第IV部の共通ライブラリ：図の設定とスペクトログラムの表示（IV-2 で定義）

  import matplotlib; matplotlib.use("Agg")      # 図をファイルに保存するだけのとき（第7章と同じ）
  from common.plotting import specshow
  specshow(X, fs, S, ax=ax, title="…")          # X は stft の結果（複素数）か、振幅・パワー

日本語のフォントは既存の章と同じ（Hiragino Sans）。
"""
import numpy as np
import matplotlib

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

__all__ = ["specshow", "to_db"]


def to_db(X, power=False, ref=None, floor_db=-120.0):
    """X（複素数・振幅・パワー）を dB にする。振幅は 20log10、パワーは 10log10。
    ref を与えないと、最大値を 0 dB にする"""
    A = np.abs(X)
    if power:
        D = 10 * np.log10(A + 1e-20)
    else:
        D = 20 * np.log10(A + 1e-20)
    D -= (D.max() if ref is None else (10 if power else 20) * np.log10(ref))
    return np.maximum(D, floor_db)


def specshow(X, fs, S, ax=None, power=False, db_range=80.0, fmax=None, title=None, cmap="magma",
             colorbar=True):
    """スペクトログラムを表示する。X は (F, T)。横軸はフレームの窓の中心の時刻 t·S/fs [s]、
    縦軸はビンの周波数 k·fs/L [Hz]（L = 2(F−1)）。色は最大値から db_range [dB] 下までを表示する。"""
    import matplotlib.pyplot as plt
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("specshow に渡す X は (F, T)。多チャネルなら X[:, :, m] のように1つ選ぶ")
    F, T = X.shape
    L = 2 * (F - 1)
    D = to_db(X, power=power)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    df, dt = fs / L, S / fs
    # 各ビン・各フレームの値を、その中心の周りの長方形で塗る
    extent = (-dt / 2, (T - 1) * dt + dt / 2, -df / 2, (F - 1) * df + df / 2)
    im = ax.imshow(D, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-db_range, vmax=0, interpolation="nearest")
    ax.set_xlabel("時間 [s]")
    ax.set_ylabel("周波数 [Hz]")
    if fmax is not None:
        ax.set_ylim(0, fmax)
    if title:
        ax.set_title(title)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="dB（最大を 0）")
    return im

In [ ]:
%%writefile common/filters.py
"""第IV部の共通ライブラリ：離散時間のシステムとフィルタ（IV-1 で定義）

NumPy だけで書いた「中身の見える」実装。どれも scipy.signal の同じ名前の関数と
数値が一致することを IV-1 の各プログラムで確かめている（速さは scipy に負ける）。

  y = conv_direct(x, h)            たたみ込みの定義どおり（二重ループ）
  C = conv_matrix(h, n)            たたみ込みの行列（テプリッツ行列）。y = C @ x
  y = conv_fft(x, h)               FFT によるたたみ込み（ゼロ詰めで円状の回り込みを防ぐ）
  w, H = freqz(b, a, n, fs)        周波数応答 H(e^{jω})（scipy.signal.freqz と同じ点）
  w, gd = group_delay(b, a, n, fs) 群遅延 [標本]
  y = lfilter(b, a, x)             差分方程式（直接形 II 転置）。zi を渡すと (y, zf)
  zi = lfilter_zi(b, a)            ステップ入力の定常状態の初期値（filtfilt で使う）
  y = filtfilt(b, a, x)            前向き＋後ろ向き（ゼロ位相）。scipy の既定と同じ端の処理
  h = firwin_lowpass(numtaps, fc, fs, window="hamming")   窓関数法の低域通過 FIR
  b, a = butter(N, fc, fs, btype="lowpass", prewarp=True) 双一次変換のバターワース
  z, p, k = butter_zpk(...)        同じものを零点・極・利得で
  y = resample_poly(x, up, down)   有理数倍のリサンプリング（scipy と同じフィルタ）
  q = quantize(x, bits, dither=None, rng=None)  一様な量子化（mid-tread）とディザ

周波数の約束：fs を渡すと周波数は Hz、渡さないと正規化角周波数 ω [rad/標本]（0〜π）。
scipy の butter・firwin は fs を渡さないと「ナイキスト周波数を 1」とする別の単位になるので、
この教科書では scipy を呼ぶときも必ず fs= を渡す。
"""
import math

import numpy as np
from scipy.signal import get_window

__all__ = ["conv_direct", "conv_matrix", "conv_fft", "freqz", "group_delay", "lfilter",
           "lfilter_zi", "filtfilt", "firwin_lowpass", "butter", "butter_zpk",
           "bilinear_zpk", "resample_poly", "quantize"]


# ---------------------------------------------------------------- たたみ込み
def conv_direct(x, h):
    """y[n] = Σ_k h[k] x[n−k]（n = 0, …, len(x)+len(h)−2）。定義どおりの二重ループ"""
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)
    y = np.zeros(len(x) + len(h) - 1)
    for n in range(len(y)):
        for k in range(len(h)):
            if 0 <= n - k < len(x):
                y[n] += h[k] * x[n - k]
    return y


def conv_matrix(h, n):
    """長さ n の入力に h をたたみ込む (n+len(h)−1) × n の行列。C[i, j] = h[i−j]（範囲外は 0）"""
    h = np.asarray(h, dtype=float)
    C = np.zeros((n + len(h) - 1, n))
    for j in range(n):
        C[j:j + len(h), j] = h
    return C


def conv_fft(x, h):
    """FFT によるたたみ込み。長さ N ≥ len(x)+len(h)−1 までゼロ詰めしてから掛け算する
    （足りないと、後ろにはみ出た分が前に回り込む＝円状たたみ込みになる）"""
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)
    n_out = len(x) + len(h) - 1
    N = 1 << (n_out - 1).bit_length()          # n_out 以上の2のべき乗
    return np.fft.irfft(np.fft.rfft(x, N) * np.fft.rfft(h, N), N)[:n_out]


# ---------------------------------------------------------------- 周波数応答
def _grid(n, fs):
    w = np.pi * np.arange(n) / n                # 0 ≤ ω < π（scipy.signal.freqz の既定と同じ点）
    return w, (w * fs / (2 * np.pi) if fs is not None else w)


def _poly_at(c, w):
    """Σ_m c[m] e^{−jωm} を、それぞれの ω で計算する"""
    m = np.arange(len(c))
    return np.exp(-1j * np.outer(w, m)) @ np.asarray(c, dtype=complex)


def freqz(b, a=1.0, n=512, fs=None):
    """H(e^{jω}) = B(e^{jω}) / A(e^{jω})。戻り値は (周波数, H)"""
    w, f = _grid(n, fs)
    return f, _poly_at(np.atleast_1d(b), w) / _poly_at(np.atleast_1d(a), w)


def group_delay(b, a=1.0, n=512, fs=None, eps=1e-10):
    """群遅延 τ(ω) = −dφ/dω [標本]。多項式 C(ω) = Σ c_m e^{−jωm} の群遅延は
    Re[ Σ m c_m e^{−jωm} / C(ω) ] なので、τ = τ_B − τ_A。
    B か A が 0 になる点（単位円の上の零点）では定義できないので nan を返す"""
    w, f = _grid(n, fs)
    out = np.zeros(n)
    for c, sign in ((np.atleast_1d(b), 1.0), (np.atleast_1d(a), -1.0)):
        C = _poly_at(c, w)
        dC = _poly_at(np.arange(len(c)) * np.asarray(c, dtype=float), w)
        bad = np.abs(C) < eps * np.max(np.abs(c))
        tau = np.full(n, np.nan)
        tau[~bad] = np.real(dC[~bad] / C[~bad])
        out = out + sign * tau
    return f, out


# ---------------------------------------------------------------- 差分方程式
def _norm_ba(b, a):
    b = np.atleast_1d(np.asarray(b, dtype=float))
    a = np.atleast_1d(np.asarray(a, dtype=float))
    if a[0] == 0:
        raise ValueError("a[0] が 0 です")
    b, a = b / a[0], a / a[0]
    K = max(len(a), len(b))
    return np.pad(b, (0, K - len(b))), np.pad(a, (0, K - len(a)))


def lfilter(b, a, x, zi=None):
    """a[0]y[n] + a[1]y[n−1] + … = b[0]x[n] + b[1]x[n−1] + … を、直接形 II 転置で計算する。
    z は K−1 個の内部の状態（K = max(len(a), len(b))）。zi を渡すと (y, 最後の状態) を返す"""
    b, a = _norm_ba(b, a)
    K = len(a)
    z = np.zeros(K - 1) if zi is None else np.array(zi, dtype=float)
    x = np.asarray(x, dtype=float)
    y = np.zeros(len(x))
    for n, xn in enumerate(x):
        yn = b[0] * xn + (z[0] if K > 1 else 0.0)
        for i in range(K - 2):
            z[i] = b[i + 1] * xn + z[i + 1] - a[i + 1] * yn
        if K > 1:
            z[K - 2] = b[K - 1] * xn - a[K - 1] * yn
        y[n] = yn
    return y if zi is None else (y, z)


def lfilter_zi(b, a):
    """入力が 1 のまま続いたときの、定常状態の内部の状態 zi（scipy.signal.lfilter_zi と同じ式）。
    z = A^T z + B（A は a のコンパニオン行列、B = b[1:] − a[1:]·b[0]）を z について解く"""
    b, a = _norm_ba(b, a)
    K = len(a)
    if K == 1:
        return np.zeros(0)
    comp = np.zeros((K - 1, K - 1))
    comp[0, :] = -a[1:]
    comp[1:, :-1] += np.eye(K - 2)
    return np.linalg.solve(np.eye(K - 1) - comp.T, b[1:] - a[1:] * b[0])


def filtfilt(b, a, x, padlen=None):
    """前向きにかけ、時間を逆にしてもう一度かける（ゼロ位相、振幅は |H|²）。
    端は scipy.signal.filtfilt の既定と同じ：padlen = 3·max(len(a), len(b)) 点を
    「点対称（odd）」に足し、ステップの定常状態から始める"""
    a1 = np.atleast_1d(a)
    b1 = np.atleast_1d(b)
    x = np.asarray(x, dtype=float)
    edge = 3 * max(len(a1), len(b1)) if padlen is None else padlen
    if len(x) <= edge:
        raise ValueError("信号が短すぎます（padlen より長くしてください）")
    left = 2 * x[0] - x[edge:0:-1]
    right = 2 * x[-1] - x[-2:-(edge + 2):-1]
    ext = np.concatenate([left, x, right])
    zi = lfilter_zi(b1, a1)
    y, _ = lfilter(b1, a1, ext, zi=zi * ext[0])
    y, _ = lfilter(b1, a1, y[::-1], zi=zi * y[-1])
    y = y[::-1]
    return y[edge:len(y) - edge] if edge > 0 else y


# ---------------------------------------------------------------- FIR の設計（窓関数法）
def firwin_lowpass(numtaps, fc, fs, window="hamming"):
    """理想の低域通過フィルタ h_d[n] = sin(ω_c m)/(π m)（m = n − (numtaps−1)/2）に窓を掛け、
    直流の利得が 1 になるように割る。scipy.signal.firwin(numtaps, fc, window=window, fs=fs) と同じ。
    窓は左右対称（get_window(..., fftbins=False)）。numtaps は奇数を勧める"""
    if not 0 < fc < fs / 2:
        raise ValueError("遮断周波数は 0 < fc < fs/2")
    nu = 2 * fc / fs                              # ω_c/π（0〜1）
    m = np.arange(numtaps) - (numtaps - 1) / 2
    h = nu * np.sinc(nu * m)                      # np.sinc(x) = sin(πx)/(πx)
    h = h * get_window(window, numtaps, fftbins=False)
    return h / np.sum(h)


# ---------------------------------------------------------------- IIR の設計（双一次変換）
def bilinear_zpk(z, p, k, fs):
    """s = 2fs(1 − z⁻¹)/(1 + z⁻¹) で、アナログの零点・極・利得をディジタルへ写す。
    s 平面の点 s₀ は z₀ = (2fs + s₀)/(2fs − s₀) へ。零点の足りない分は z = −1（ω = π）に置く"""
    z = np.atleast_1d(np.asarray(z, dtype=complex))
    p = np.atleast_1d(np.asarray(p, dtype=complex))
    c = 2.0 * fs
    zd = (c + z) / (c - z)
    pd = (c + p) / (c - p)
    zd = np.concatenate([zd, -np.ones(len(p) - len(z))])
    kd = k * np.real(np.prod(c - z) / np.prod(c - p))
    return zd, pd, kd


def butter_zpk(N, fc, fs, btype="lowpass", prewarp=True):
    """N 次のバターワース（低域・高域）。アナログの原型の極 e^{jπ(2k+N−1)/(2N)}（左半面）を
    遮断角周波数 Ωc に広げてから双一次変換する。prewarp=True なら Ωc = 2fs·tan(π fc/fs)
    （ディジタルで −3 dB の点がちょうど fc に来る）、False なら Ωc = 2π fc（ずれる）"""
    k_idx = np.arange(1, N + 1)
    proto = np.exp(1j * np.pi * (2 * k_idx + N - 1) / (2 * N))
    Wc = 2 * fs * np.tan(np.pi * fc / fs) if prewarp else 2 * np.pi * fc
    if btype == "lowpass":
        z, p, k = np.array([]), Wc * proto, Wc ** N
    elif btype == "highpass":
        z, p, k = np.zeros(N), Wc / proto, 1.0
    else:
        raise ValueError("btype は lowpass か highpass")
    return bilinear_zpk(z, p, np.real(k), fs)


def butter(N, fc, fs, btype="lowpass", prewarp=True):
    """butter_zpk を多項式の係数 (b, a) にする。次数が高い（目安 8 以上）・遮断周波数が低いときは
    係数の丸め誤差で不安定になりやすいので、実際の処理では scipy の output="sos" を使う（IV-1-3-11）"""
    z, p, k = butter_zpk(N, fc, fs, btype, prewarp)
    return np.real(k * np.poly(z)), np.real(np.poly(p))


# ---------------------------------------------------------------- リサンプリング
def resample_poly(x, up, down, window=("kaiser", 5.0)):
    """標本化周波数を up/down 倍にする。(1) 標本の間に up−1 個の 0 を入れ、
    (2) 低域通過 FIR（遮断 = 元と新しいもののナイキスト周波数の低いほう）をかけて up 倍し、
    (3) down 個に1個を取り出す。フィルタは scipy.signal.resample_poly の既定と同じ
    （長さ 2·10·max(up, down)+1、カイザー窓 β = 5）。遅れ（フィルタの中心）は取り除く。

    (1)〜(3) をそのまま計算すると、0 との掛け算と、捨てる出力の計算がほとんどで遅い
    （44.1 k → 16 k では up = 160 なので 99% 以上がむだ）。そこで、取り出す出力 m だけを
        y[m] = Σ_i x[i]·h[m·down + half − i·up]（h の範囲に入る i だけ）
    と計算する（ポリフェーズ実装の考え方。結果は (1)〜(3) と同じ）"""
    g = math.gcd(up, down)
    up, down = up // g, down // g
    x = np.asarray(x, dtype=float)
    if up == down == 1:
        return x.copy()
    r = max(up, down)
    half = 10 * r
    fs_up = 2.0 * r                                # 「ナイキスト周波数 = r」とした単位で設計する
    h = firwin_lowpass(2 * half + 1, 1.0, fs_up, window=window) * up
    n_out = -(-len(x) * up // down)                # 切り上げ
    pos = np.arange(n_out) * down + half           # 出力 m が、ゼロを入れた列のどこに当たるか（＋遅れ half）
    J = (2 * half) // up + 1                       # 1 つの出力に効く入力の数（最大）
    i = pos[:, None] // up - np.arange(J)[None, :]  # 効く入力の番号 i（新しい順）
    idx = pos[:, None] - i * up                    # そのときの h の番号（0〜2·half に入るものだけ使う）
    ok = (idx <= 2 * half) & (i >= 0) & (i < len(x))
    terms = np.where(ok, h[np.clip(idx, 0, 2 * half)] * x[np.clip(i, 0, len(x) - 1)], 0.0)
    return terms.sum(axis=1)


# ---------------------------------------------------------------- 量子化
def quantize(x, bits, dither=None, rng=None):
    """−1〜1 の信号を bits ビットで量子化する（mid-tread：0 が目盛りの上）。
    目盛り Δ = 2/2^bits、値は −1 〜 1−Δ に止める。
    dither="rect"：±Δ/2 の一様乱数、"tpdf"：±Δ の三角分布の乱数（一様2つの和）を、丸める前に足す"""
    x = np.asarray(x, dtype=float)
    d = 2.0 / 2 ** bits
    if dither is not None:
        rng = np.random.default_rng() if rng is None else rng
        if dither == "rect":
            x = x + rng.uniform(-d / 2, d / 2, x.shape)
        elif dither == "tpdf":
            x = x + rng.uniform(-d / 2, d / 2, x.shape) + rng.uniform(-d / 2, d / 2, x.shape)
        else:
            raise ValueError("dither は None・'rect'・'tpdf'")
    return np.clip(d * np.round(x / d), -1.0, 1.0 - d)

In [ ]:
%%writefile common/array.py
"""第IV部の共通ライブラリ：マイクロホンアレイ（IV-5 で定義。IV-6・最終演習K でも使う）

約束（IV-5 の本文の記号と同じ）
  mics      : マイクの位置。形 (D, M)（D = 2 か 3、単位 m）。pyroomacoustics の R と同じ並び
  azimuth   : 方位角 θ [rad]。x 軸から反時計回り（pyroomacoustics と同じ）。
              直線アレイを x 軸に沿って置くと、θ = 90° が正面（ブロードサイド）、0°・180° が軸の方向（エンドファイア）
  u(θ)      : 原点から音源の方向を向く単位ベクトル [cosθ, sinθ(, 0)]
  遅延      : 平面波（遠方場）では、マイク m に届く時刻は原点より τ_m = −u·r_m / c だけ遅い
  ステアリングベクトル a_f(θ) の成分 a_m = exp(−j2πf τ_m) = exp(+j2πf u·r_m / c)
              （STFT の e^{−j2πkm/L} の約束で「τ 遅れると e^{−j2πfτ} が掛かる」。pyroomacoustics.doa と同じ符号）
  X         : 観測の STFT。形 (F, T, M)（common.stft.stft の多チャネルの出力）
  重み w     : 出力 y_ft = w_fᴴ x_ft = Σ_m conj(w_fm) x_ftm。W の形は (F, M)

関数
  linear_array, circular_array, unit_vector, far_field_delays, steering_vector,
  scm, ds_weights, mvdr_weights, apply_weights, beam_pattern, white_noise_gain,
  gcc_phat, tdoa_to_azimuth, music_spectrum, srp_phat_spectrum,
  sfo_resample, sfo_phase
"""
import numpy as np

C_SOUND = 343.0          # 音速 [m/s]（約 20 ℃。pyroomacoustics の既定と同じ）

__all__ = ["C_SOUND", "linear_array", "circular_array", "unit_vector", "far_field_delays",
           "steering_vector", "scm", "ds_weights", "mvdr_weights", "apply_weights", "beam_pattern",
           "white_noise_gain", "gcc_phat", "tdoa_to_azimuth", "music_spectrum", "srp_phat_spectrum",
           "sfo_resample", "sfo_phase"]


# ---------------------------------------------------------------- 配置
def linear_array(M, d, center=(0.0, 0.0), angle=0.0):
    """間隔 d [m] の M 本の直線アレイ。中心が center、向きが angle [rad]（x 軸から）。形 (2, M)
    （pyroomacoustics.linear_2D_array と同じ）"""
    u = np.array([np.cos(angle), np.sin(angle)])
    k = np.arange(M) - (M - 1) / 2
    return np.asarray(center, float)[:, None] + d * k[None, :] * u[:, None]


def circular_array(M, radius, center=(0.0, 0.0), phi0=0.0):
    """半径 radius [m] の M 本の円形アレイ。形 (2, M)（pyroomacoustics.circular_2D_array と同じ）"""
    phi = phi0 + 2 * np.pi * np.arange(M) / M
    return np.asarray(center, float)[:, None] + radius * np.vstack([np.cos(phi), np.sin(phi)])


def unit_vector(azimuth, dim=2):
    """方位角（スカラーか (K,)）の単位ベクトル。形 (dim,) か (dim, K)"""
    az = np.asarray(azimuth, float)
    u = [np.cos(az), np.sin(az)] + ([np.zeros_like(az)] if dim == 3 else [])
    return np.array(u)


# ---------------------------------------------------------------- ステアリングベクトル
def far_field_delays(mics, azimuth, c=C_SOUND):
    """平面波の到来時刻の遅れ τ_m = −u·r_m / c [s]（原点を基準）。形 (M,) か (K, M)"""
    u = unit_vector(azimuth, dim=mics.shape[0])
    return -(u.T @ mics) / c


def steering_vector(mics, freqs, azimuth=None, source=None, c=C_SOUND, amplitude=False):
    """ステアリングベクトル。
    遠方場（azimuth を与える）：a_m = exp(−j2πf τ_m)、τ_m = −u·r_m / c。
        azimuth がスカラーなら形 (F, M)、(K,) なら (F, K, M)
    近距離場（source に音源の位置 (D,) を与える）：アレイの中心 c̄ からの距離 ρ_0 を基準に
        τ_m = (|p − r_m| − |p − c̄|) / c。amplitude=True なら距離の減衰 ρ_0/|p − r_m| も掛ける。形 (F, M)
    """
    f = np.atleast_1d(np.asarray(freqs, float))
    if (azimuth is None) == (source is None):
        raise ValueError("azimuth か source のどちらか一方を与える")
    if azimuth is not None:
        tau = far_field_delays(mics, azimuth, c)                      # (M,) か (K, M)
        return np.exp(-2j * np.pi * f.reshape((-1,) + (1,) * tau.ndim) * tau)
    p = np.asarray(source, float).reshape(-1, 1)[: mics.shape[0]]
    dist = np.linalg.norm(p - mics, axis=0)                          # (M,)
    ref = np.linalg.norm(p[:, 0] - mics.mean(axis=1))
    a = np.exp(-2j * np.pi * f[:, None] * (dist - ref)[None, :] / c)
    return a * (ref / dist)[None, :] if amplitude else a


# ---------------------------------------------------------------- 空間相関行列とビームフォーマ
def scm(X):
    """空間相関行列 R_f = (1/T) Σ_t x_ft x_ftᴴ（複素正規分布の共分散の最尤推定。IV-4-3-4）。
    X (F, T, M) → (F, M, M)"""
    return np.einsum("ftm,ftn->fmn", X, X.conj()) / X.shape[1]


def ds_weights(a):
    """遅延和ビームフォーマ w = a / M（目的の方向の利得が 1）。a (F, M) → (F, M)"""
    return a / a.shape[-1]


def mvdr_weights(a, R, loading=0.0):
    """MVDR（最小分散無歪み応答）w = R⁻¹a / (aᴴR⁻¹a)。
    loading > 0 なら対角装荷：R + ε I、ε = loading × tr(R)/M（周波数ごと）。a (F, M)、R (F, M, M)"""
    R = np.asarray(R, complex)
    M = R.shape[-1]
    if loading > 0:
        eps = loading * np.real(np.trace(R, axis1=-2, axis2=-1)) / M
        R = R + eps[:, None, None] * np.eye(M)[None]
    Ria = np.linalg.solve(R, a[..., None])[..., 0]                    # R⁻¹a
    den = np.einsum("fm,fm->f", a.conj(), Ria)                       # aᴴR⁻¹a（実数で正）
    return Ria / den[:, None]


def apply_weights(W, X):
    """y_ft = w_fᴴ x_ft。W (F, M)、X (F, T, M) → (F, T)"""
    return np.einsum("fm,ftm->ft", W.conj(), X)


def beam_pattern(W, mics, freqs, azimuths, c=C_SOUND):
    """ビームパターン B_f(θ) = w_fᴴ a_f(θ)（複素数）。W (F, M) → (F, K)"""
    A = steering_vector(mics, freqs, azimuth=np.asarray(azimuths, float), c=c)   # (F, K, M)
    return np.einsum("fm,fkm->fk", W.conj(), A)


def white_noise_gain(W, a):
    """白色雑音利得 WNG = |wᴴa|² / (wᴴw)（マイクごとに独立な雑音を何倍に抑えるか。DS で M）"""
    num = np.abs(np.einsum("fm,fm->f", W.conj(), a)) ** 2
    return num / np.real(np.einsum("fm,fm->f", W.conj(), W))


# ---------------------------------------------------------------- 到来方向推定
def gcc_phat(x1, x2, fs, max_tau=None, interp=1):
    """GCC-PHAT で、x2 が x1 より何秒遅れているか（τ > 0 なら x2 が遅い）を推定する。
    G(ω) = X2(ω) X1*(ω) / |X2(ω) X1*(ω)|、cc(τ) = IDFT[G]（長さを interp 倍にして細かくする）
    戻り値: tau [s], lags [s], cc（lags の順に並べた相互相関）"""
    x1 = np.asarray(x1, float)
    x2 = np.asarray(x2, float)
    n = len(x1) + len(x2)                                            # 円状のたたみ込みにならない長さ
    n += n % 2
    G = np.fft.rfft(x2, n) * np.conj(np.fft.rfft(x1, n))
    G /= np.abs(G) + 1e-20
    cc = np.fft.irfft(G, interp * n)
    half = interp * n // 2
    cc = np.concatenate([cc[-half:], cc[:half]])                     # ラグ −half 〜 half−1
    lags = (np.arange(-half, half)) / (interp * fs)
    if max_tau is not None:
        keep = np.abs(lags) <= max_tau
        cc, lags = cc[keep], lags[keep]
    return float(lags[np.argmax(cc)]), lags, cc


def tdoa_to_azimuth(tau, d, c=C_SOUND):
    """2本のマイク（x 軸上、マイク2 が +x 側に d）の時間差 τ（マイク2 の遅れ）から方位角 θ [rad]。
    τ = −d cosθ / c なので θ = arccos(−cτ/d)（0〜π。表と裏は区別できない）"""
    return float(np.arccos(np.clip(-c * tau / d, -1.0, 1.0)))


def music_spectrum(X, mics, freqs, azimuths, n_src=1, normalize=False, c=C_SOUND):
    """広帯域 MUSIC の空間スペクトル（周波数ごとの 1/|aᴴ E_n E_nᴴ a| の平均）。
    X (F', T, M) は使う周波数のビンだけを取り出した STFT、freqs (F',) はその周波数 [Hz]。
    normalize=True なら周波数ごとに最大を 1 にしてから平均（pyroomacoustics の NormMUSIC）"""
    R = scm(X)
    _, V = np.linalg.eigh(R)                                         # 固有値の小さい順
    En = V[:, :, : X.shape[2] - n_src]                               # 雑音部分空間 (F', M, M−N)
    A = steering_vector(mics, freqs, azimuth=azimuths, c=c)          # (F', K, M)
    proj = np.einsum("fkm,fmj->fkj", A.conj(), En)                   # aᴴ E_n
    P = 1.0 / np.sum(np.abs(proj) ** 2, axis=2)                      # (F', K)
    if normalize:
        P = P / P.max(axis=1, keepdims=True)
    return P.mean(axis=0)


def srp_phat_spectrum(X, mics, freqs, azimuths, c=C_SOUND):
    """SRP-PHAT：振幅を 1 にそろえた観測 x̃ = x/|x|（成分ごと）に遅延和をかけた出力のパワー
    |aᴴ x̃|² を、フレームと周波数で平均したもの。X (F', T, M)"""
    Xn = X / np.maximum(np.abs(X), 1e-14)
    A = steering_vector(mics, freqs, azimuth=azimuths, c=c)          # (F', K, M)
    out = np.einsum("fkm,ftm->fkt", A.conj(), Xn)
    return np.mean(np.abs(out) ** 2, axis=(0, 2))


# ---------------------------------------------------------------- 非同期録音（標本化周波数のずれ）
def sfo_resample(x, eps, pad=4096):
    """標本化周波数が fs(1+eps) の機器で録った信号を作る：y[n] = x_c(n / (fs(1+eps)))。
    FFT による帯域制限の補間（scipy.signal.resample）。eps はサンプル数の都合で少し丸める。
    戻り値: y（長さは x と同じ）, 実際に使った eps"""
    from scipy.signal import resample
    x = np.asarray(x, float)
    N = len(x) + pad                                                 # 端の回り込みを防ぐゼロ
    if eps != 0:                                                     # N·eps が整数に近くなる N を選ぶ（丸めの誤差を小さく）
        cand = np.arange(N, N + int(2 / abs(eps)) + 2)
        N = int(cand[np.argmin(np.abs(cand * eps - np.round(cand * eps)))])
        pad = N - len(x)
    num = int(round(N * (1 + eps)))
    y = resample(np.concatenate([x, np.zeros(pad)]), num)
    return y[: len(x)], num / N - 1


def sfo_phase(F, T, L, S, eps):
    """フレーム t（中心の時刻 tS/fs）での、ずれ eps による位相の回転 exp(−j2π k eps t S / L)。形 (F, T)。
    標本化周波数が fs(1+eps) のチャネルは、時刻 τ で eps·τ だけ遅れて見える（IV-5-3-10）。
    補正は、そのチャネルの STFT にこの共役を掛ける"""
    k = np.arange(F)[:, None]
    t = np.arange(T)[None, :]
    return np.exp(-2j * np.pi * k * eps * t * S / L)

In [ ]:
%%writefile common/room.py
"""第IV部の共通ライブラリ：音と部屋の音響学（IV-3 で定義）

約束（IV-3 の本文の記号と同じ）
  dim    : 直方体の部屋の大きさ [Lx, Ly, Lz]（m）。原点は部屋の角、壁は x=0, x=Lx, y=0, y=Ly, z=0, z=Lz
  alpha  : 壁の（エネルギーの）吸音率 α。スカラーか、6 つの壁の値 [x=0, x=Lx, y=0, y=Ly, z=0, z=Lz]
           （pyroomacoustics の壁の順 west, east, south, north, floor, ceiling と同じ）
  beta   : 音圧の反射係数 β = sqrt(1 − α)（角度と周波数によらないとみなす）
  h      : インパルス応答（1次元の配列）。fs [Hz] で標本化
  EDC    : シュレーダーの逆積分 E(t) = Σ_{τ ≥ t} h²(τ)（エネルギーの減衰曲線）
  T60    : エネルギーが 60 dB 下がる時間 [s]。T20・T30 は −5〜−25 dB・−5〜−35 dB の傾きから 60 dB に外挿したもの

関数
  音波      : speed_of_sound, spl_db, a_weighting_db
  部屋      : room_volume_surface, rt60_sabine, rt60_eyring, sabine_absorption, critical_distance,
              schroeder_frequency, room_modes
  鏡像法    : image_sources, fractional_delay, ism_rir
  残響時間  : schroeder, edc_db, decay_fit, lundeby, rt60_from_ir, drr_db, clarity_db, octave_band, polack_rir
  測定信号  : ess, ess_deconvolve, harmonic_delay, tsp, tsp_deconvolve
  指向性    : first_order_pattern, directivity_factor
"""
import numpy as np
from scipy.signal import butter, sosfilt

P_REF = 20e-6            # 音圧の基準 20 µPa（dB SPL の 0 dB）
C_SOUND = 343.0          # 音速 [m/s]（約 20 ℃。pyroomacoustics の既定、common/array.py と同じ）

__all__ = ["P_REF", "C_SOUND", "speed_of_sound", "spl_db", "a_weighting_db",
           "room_volume_surface", "rt60_sabine", "rt60_eyring", "sabine_absorption", "critical_distance",
           "schroeder_frequency", "room_modes",
           "image_sources", "fractional_delay", "ism_rir",
           "schroeder", "edc_db", "decay_fit", "lundeby", "rt60_from_ir", "drr_db", "clarity_db",
           "octave_band", "polack_rir",
           "ess", "ess_deconvolve", "harmonic_delay", "tsp", "tsp_deconvolve",
           "first_order_pattern", "directivity_factor"]


# ---------------------------------------------------------------- 音波
def speed_of_sound(temp_c=20.0):
    """乾いた空気の音速 c = sqrt(γRT/M) = 331.3·sqrt(1 + θ/273.15) [m/s]（θ は ℃）。
    湿度の効果は小さい（20 ℃ で湿度 0→100% にして約 +0.3%。本文の 3-1 節）"""
    return 331.3 * np.sqrt(1.0 + np.asarray(temp_c, dtype=float) / 273.15)


def spl_db(p_rms):
    """音圧の実効値 p_rms [Pa] → 音圧レベル [dB SPL]（基準 20 µPa）"""
    return 20 * np.log10(np.asarray(p_rms, dtype=float) / P_REF)


def a_weighting_db(f):
    """A 特性の重み [dB]（IEC 61672-1 の式。1 kHz で 0 dB）"""
    f2 = np.asarray(f, dtype=float) ** 2
    ra = (12194.0 ** 2 * f2 ** 2) / ((f2 + 20.6 ** 2) * np.sqrt((f2 + 107.7 ** 2) * (f2 + 737.9 ** 2))
                                    * (f2 + 12194.0 ** 2))
    return 20 * np.log10(ra) + 2.00


# ---------------------------------------------------------------- 部屋（拡散音場の式）
def _walls(dim):
    Lx, Ly, Lz = map(float, dim)
    return np.array([Ly * Lz, Ly * Lz, Lx * Lz, Lx * Lz, Lx * Ly, Lx * Ly])   # 6 つの壁の面積


def room_volume_surface(dim):
    """直方体の部屋の体積 V [m³] と表面積 S [m²]"""
    return float(np.prod(dim)), float(_walls(dim).sum())


def _mean_alpha(dim, alpha):
    a = np.broadcast_to(np.asarray(alpha, dtype=float), (6,))
    Sw = _walls(dim)
    return float(np.sum(Sw * a) / Sw.sum())


def rt60_sabine(dim, alpha, c=C_SOUND, m_air=0.0):
    """サビンの式 T = 24 ln10 · V / (c (Σ S_i α_i + 4 m V))（m_air は空気の吸収 [1/m]）"""
    V, S = room_volume_surface(dim)
    A = S * _mean_alpha(dim, alpha)                     # 等価吸音面積 [m²]
    return 24 * np.log(10) * V / (c * (A + 4 * m_air * V))


def rt60_eyring(dim, alpha, c=C_SOUND, m_air=0.0):
    """アイリングの式 T = 24 ln10 · V / (c (−S ln(1 − ᾱ) + 4 m V))（ᾱ は面積で重みをつけた平均）"""
    V, S = room_volume_surface(dim)
    return 24 * np.log(10) * V / (c * (-S * np.log(1.0 - _mean_alpha(dim, alpha)) + 4 * m_air * V))


def sabine_absorption(rt60, dim, c=C_SOUND):
    """サビンの式を逆に解いた、すべての壁に共通の吸音率 α = 24 ln10 · V / (c S T)
    （pyroomacoustics.inverse_sabine の1つ目の戻り値と同じ）"""
    V, S = room_volume_surface(dim)
    return 24 * np.log(10) * V / (c * S * rt60)


def critical_distance(dim, rt60, Q=1.0, c=C_SOUND):
    """臨界距離 r_c = sqrt(Q A / (16π))。A はサビンの式から逆算した等価吸音面積 24 ln10 V/(c T)"""
    V, _ = room_volume_surface(dim)
    A = 24 * np.log(10) * V / (c * rt60)
    return np.sqrt(Q * A / (16 * np.pi))


def schroeder_frequency(rt60, V):
    """シュレーダー周波数の目安 f_S ≈ 2000 sqrt(T60 / V) [Hz]（V は m³）"""
    return 2000.0 * np.sqrt(rt60 / V)


def room_modes(dim, fmax, c=C_SOUND):
    """剛い壁の直方体の部屋の固有周波数 f = (c/2) sqrt((nx/Lx)² + (ny/Ly)² + (nz/Lz)²) のうち fmax 以下。
    戻り値は (f, nx, ny, nz) を並べた (K, 4) の配列（f の小さい順。(0,0,0) は除く）"""
    Lx, Ly, Lz = map(float, dim)
    n = [int(np.floor(2 * fmax * L / c)) for L in (Lx, Ly, Lz)]
    nx, ny, nz = np.meshgrid(*[np.arange(k + 1) for k in n], indexing="ij")
    f = c / 2 * np.sqrt((nx / Lx) ** 2 + (ny / Ly) ** 2 + (nz / Lz) ** 2)
    sel = (f <= fmax) & (f > 0)
    out = np.c_[f[sel], nx[sel], ny[sel], nz[sel]]
    return out[np.argsort(out[:, 0], kind="stable")]


# ---------------------------------------------------------------- 鏡像法（Allen & Berkley 1979）
def image_sources(dim, src, max_order):
    """直方体の部屋の鏡像音源。反射の次数 |nx|+|ny|+|nz| ≤ max_order のものをすべて返す。
    軸 d の番号 n の鏡像の座標は n·L_d + (s_d（n が偶数）か L_d − s_d（n が奇数）)。
    戻り値：pos (K, 3)、order (K,)、hits (K, 6)（6 つの壁 [x=0, x=Lx, y=0, …] で反射した回数）"""
    L = np.asarray(dim, dtype=float)
    s = np.asarray(src, dtype=float)
    N = int(max_order)
    r = np.arange(-N, N + 1)
    nx, ny, nz = np.meshgrid(r, r, r, indexing="ij")
    n = np.c_[nx.ravel(), ny.ravel(), nz.ravel()]
    n = n[np.abs(n).sum(axis=1) <= N]
    odd = (np.abs(n) % 2 == 1)
    pos = n * L + np.where(odd, L - s, s)
    # 反射の回数：n > 0 は x=L 側が ceil(n/2) 回、x=0 側が floor(n/2) 回。n < 0 は逆
    lo = np.where(n > 0, n // 2, (-n + 1) // 2)          # x=0（y=0、z=0）側の壁
    hi = np.where(n > 0, (n + 1) // 2, (-n) // 2)        # x=L 側の壁
    hits = np.empty((len(n), 6), dtype=int)
    hits[:, 0::2] = lo
    hits[:, 1::2] = hi
    return pos, np.abs(n).sum(axis=1), hits


def fractional_delay(frac, fdl=81):
    """整数でない遅れ frac（0 ≤ frac < 1 の配列）を表す長さ fdl のハン窓つき sinc。
    行 k の i 番目は、時刻 floor(t)+i−(fdl−1)/2 に足す値 hann[i]·sinc(i − (fdl−1)/2 − frac)
    （窓は整数の位置に固定。pyroomacoustics の rir_builder と同じ決め方）"""
    frac = np.atleast_1d(np.asarray(frac, dtype=float))
    half = (fdl - 1) // 2
    k = np.arange(-half, half + 1)
    win = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(fdl) / (fdl - 1))
    return win[None, :] * np.sinc(k[None, :] - frac[:, None])


def ism_rir(dim, src, mic, fs, alpha, max_order, c=C_SOUND, fdl=81, pra_compat=False):
    """鏡像法の部屋のインパルス応答（自前の実装）。
    振幅 = Π β_w^(反射の回数) / (4π d)、到着時刻 = d / c、分数遅延は長さ fdl のハン窓つき sinc。
    pra_compat=True にすると pyroomacoustics 0.10.1 と同じ約束にする：
      ・1/(4π) を掛けない（振幅は Πβ / d）・全体を (fdl−1)/2 標本だけ遅らせる
    （高域通過フィルタ rir_hpf は掛けないので、照合では pra.constants.set("rir_hpf_enable", False) にする）
    戻り値：h（1次元）、info（鏡像の pos・order・dist・amp・t の辞書）"""
    pos, order, hits = image_sources(dim, src, max_order)
    beta = np.sqrt(1.0 - np.broadcast_to(np.asarray(alpha, dtype=float), (6,)))
    amp_r = np.prod(beta[None, :] ** hits, axis=1)
    dist = np.linalg.norm(pos - np.asarray(mic, dtype=float)[None, :], axis=1)
    amp = amp_r / dist if pra_compat else amp_r / (4 * np.pi * dist)
    half = (fdl - 1) // 2
    t = dist / c + (half / fs if pra_compat else 0.0)
    x = t * fs
    xi = np.floor(x).astype(int)
    fr = x - xi
    n_len = int(np.ceil(x.max() + half + 1)) + 1
    off = half if not pra_compat else 0                   # 先頭より前にはみ出さないように前を空ける
    h = np.zeros(n_len + off)
    taps = fractional_delay(fr, fdl) * amp[:, None]
    idx = xi[:, None] + np.arange(-half, half + 1)[None, :] + off
    np.add.at(h, idx.ravel(), taps.ravel())
    h = h[off:]                                           # 時刻 0 を h[0] にそろえる（直前のはみ出しは捨てる）
    return h, {"pos": pos, "order": order, "dist": dist, "amp": amp, "t": t}


# ---------------------------------------------------------------- 残響時間
def schroeder(h, t_end=None, noise_power=0.0, compensation=0.0):
    """シュレーダーの逆積分 E[n] = Σ_{k=n}^{end} (h[k]² − noise_power) + compensation（線形のエネルギー）。
    t_end（標本）で打ち切る。ノイズの扱い：noise_power を引く（Chu）、打ち切り＋補正（Lundeby）"""
    p = np.asarray(h, dtype=float) ** 2 - noise_power
    if t_end is not None:
        p = p[:int(t_end)]
    return np.cumsum(p[::-1])[::-1] + compensation


def edc_db(E):
    """エネルギーの減衰曲線を dB にする（先頭を 0 dB。0 以下は nan）"""
    E = np.asarray(E, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        D = 10 * np.log10(E / E[0])
    D[~np.isfinite(D)] = np.nan
    return D


def decay_fit(D, fs, top=-5.0, bottom=-35.0):
    """dB の減衰曲線 D のうち、はじめて top を下回った標本から、はじめて bottom を下回る直前までに
    直線を最小二乗で当てはめ、60 dB に外挿した時間 T = −60 / 傾き [s] を返す。
    （pyroomacoustics の measure_rt60(decay_db=30) と同じ範囲の決め方。下回らなければ nan）
    戻り値：辞書 T, slope [dB/s], intercept [dB], i0, i1"""
    D = np.asarray(D, dtype=float)
    below_top = np.where(D < top)[0]
    below_bot = np.where(D < bottom)[0]
    if len(below_top) == 0 or len(below_bot) == 0:
        return {"T": np.nan, "slope": np.nan, "intercept": np.nan, "i0": None, "i1": None}
    i0, i1 = int(below_top[0]), int(below_bot[0])
    t = np.arange(i0, i1) / fs
    A = np.c_[t, np.ones_like(t)]
    (slope, icpt), *_ = np.linalg.lstsq(A, D[i0:i1], rcond=None)
    return {"T": -60.0 / slope, "slope": slope, "intercept": icpt, "i0": i0, "i1": i1}


def lundeby(h, fs, block_ms=10.0, n_iter=5, margin_db=10.0):
    """雑音の床と、減衰の直線が雑音に交わる時刻（クロスポイント）を反復で求める（Lundeby ら 1995 の考え方を簡略化）。
      1. h² を block_ms ごとに平均した包絡（dB）を作る
      2. 雑音のパワーを、はじめは最後の 10% の平均とする
      3. 包絡の −5 dB から「雑音 + margin_db」までに直線を当てはめ、雑音の高さと交わる時刻 t_c を求める
      4. 雑音を「t_c から、直線がさらに margin_db 下がる時刻より後ろ」（最低でも最後の 10%）で測り直し、3 に戻る
    戻り値：辞書 t_cross（標本）、noise（1 標本あたりのパワー）、comp（t_c より後ろのエネルギーの補正）、slope_db（dB/標本）"""
    p = np.asarray(h, dtype=float) ** 2
    n = len(p)
    i_pk = int(np.argmax(p))
    B = max(1, int(round(block_ms * 1e-3 * fs)))
    nb = (n - i_pk) // B
    env = p[i_pk:i_pk + nb * B].reshape(nb, B).mean(axis=1)
    tb = i_pk + (np.arange(nb) + 0.5) * B                     # 各ブロックの中心の標本
    noise = p[int(0.9 * n):].mean()
    t_c = n - 1
    slope = np.nan
    ref = env.max()
    for _ in range(n_iter):
        Ed = 10 * np.log10(env / ref + 1e-300)
        nd = 10 * np.log10(noise / ref + 1e-300)
        i0 = np.where(Ed < -5.0)[0]
        i1 = np.where(Ed < nd + margin_db)[0]
        if len(i0) == 0 or len(i1) == 0 or i1[0] <= i0[0] + 2:
            break
        a, b = i0[0], i1[0]
        slope, icpt = np.polyfit(tb[a:b], Ed[a:b], 1)            # dB/標本
        if slope >= 0:
            break
        t_c = int(np.clip((nd - icpt) / slope, i_pk + 1, n - 1))
        t_noise = int(min(t_c + margin_db / -slope, 0.9 * n))     # 直線がさらに margin_db 下がる時刻
        noise = p[t_noise:].mean()
    comp = np.nan
    if np.isfinite(slope) and slope < 0:
        tau = 10 * np.log10(np.e) / -slope                        # パワーの時定数 [標本]
        icpt_lin = ref * 10 ** ((slope * t_c + icpt) / 10)        # 直線の t_c でのパワー
        comp = icpt_lin * tau                                    # ∫_{t_c}^∞ P e^{−(t−t_c)/τ} dt
    return {"t_cross": t_c, "noise": noise, "comp": comp, "slope_db": slope}


def rt60_from_ir(h, fs, method="T30", noise="lundeby", start_at_peak=True):
    """インパルス応答から残響時間を求める。
    method : "EDT"（0〜−10 dB）、"T20"（−5〜−25）、"T30"（−5〜−35）
    noise  : "none"（そのまま最後まで積分）、"chu"（雑音のパワーを引く）、
             "lundeby"（雑音のパワーを引き、クロスポイントで打ち切り、その先を指数の減衰で補う）
    戻り値：辞書 T, D（dB の EDC）, fit（decay_fit の結果）, info（lundeby の結果か None）"""
    rng = {"EDT": (0.0, -10.0), "T20": (-5.0, -25.0), "T30": (-5.0, -35.0)}[method]
    h = np.asarray(h, dtype=float)
    if start_at_peak:
        h = h[int(np.argmax(np.abs(h))):]
    info = None
    if noise == "none":
        E = schroeder(h)
    elif noise == "chu":
        nz = np.mean(h[int(0.9 * len(h)):] ** 2)
        E = schroeder(h, noise_power=nz)
    elif noise == "lundeby":
        info = lundeby(h, fs)
        E = schroeder(h, t_end=info["t_cross"], noise_power=info["noise"],
                      compensation=np.nan_to_num(info["comp"]))
    else:
        raise ValueError(noise)
    D = edc_db(E)
    top = rng[0] if method != "EDT" else -1e-9          # EDT は 0 dB から（先頭の標本から）
    fit = decay_fit(D, fs, top=top, bottom=rng[1])
    return {"T": fit["T"], "D": D, "fit": fit, "info": info}


def drr_db(h, fs, win_ms=2.5):
    """直接音と残響音のエネルギー比 DRR [dB]。直接音は最大の山の前後 ±win_ms、残りを残響とする"""
    h = np.asarray(h, dtype=float)
    i = int(np.argmax(np.abs(h)))
    w = int(round(win_ms * 1e-3 * fs))
    e = h ** 2
    direct = e[max(0, i - w):i + w + 1].sum()
    return 10 * np.log10(direct / (e.sum() - direct))


def clarity_db(h, fs, te_ms=50.0):
    """明瞭度 C50（te_ms = 50）・C80：直接音の到着から te_ms までと、それより後ろのエネルギーの比 [dB]"""
    h = np.asarray(h, dtype=float)
    i = int(np.argmax(np.abs(h)))
    k = i + int(round(te_ms * 1e-3 * fs))
    e = h ** 2
    return 10 * np.log10(e[i:k].sum() / e[k:].sum())


def octave_band(h, fs, fc, order=3, reverse=True):
    """中心 fc [Hz] の1オクターブの帯域通過（バターワース、帯域 fc/√2〜fc·√2）。
    reverse=True なら時間を逆にしてかけてから戻す（フィルタ自身の響きが減衰の後ろに付かない）"""
    sos = butter(order, [fc / np.sqrt(2), fc * np.sqrt(2)], btype="bandpass", fs=fs, output="sos")
    h = np.asarray(h, dtype=float)
    return sosfilt(sos, h[::-1])[::-1] if reverse else sosfilt(sos, h)


def polack_rir(rt60, fs, sec, rng=None, direct=True, pnr_db=None):
    """Polack の確率モデルの人工インパルス応答：白色のガウス雑音 × 指数の減衰（エネルギーが rt60 で 60 dB 下がる）。
    direct=True なら先頭に直接音（大きさ 1）を置く。pnr_db を与えると、残響の始まりのパワーより pnr_db [dB] 小さい
    白色雑音（測定の雑音の床）を全体に足す。戻り値：h（長さ sec·fs）"""
    rng = np.random.default_rng(rng)
    n = int(sec * fs)
    t = np.arange(n) / fs
    h = rng.standard_normal(n) * 10 ** (-3 * t / rt60) * 0.05
    if direct:
        h[0] = 1.0
    if pnr_db is not None:
        h = h + rng.standard_normal(n) * 0.05 * 10 ** (-pnr_db / 20)
    return h


# ---------------------------------------------------------------- 測定信号
def ess(f1, f2, T, fs, fade=(0.02, 0.01)):
    """指数スイープ（ESS、Farina 2000）と、その逆フィルタ。
      x(t) = sin(2π f1 T/R · (e^{tR/T} − 1))、R = ln(f2/f1)、瞬時周波数 f(t) = f1 e^{tR/T}
      逆フィルタ v(t) = x(T − t) · e^{−tR/T}（時間を逆にし、振幅を周波数に比例させて 6 dB/oct で下げる）
    fade = (始まり, 終わり) [s] のハン窓の半分で、急な立ち上がり（クリック）を防ぐ。
    逆フィルタは、x * v の中央の周波数 sqrt(f1 f2) での大きさが 1 になるように規格化する。
    戻り値：x, v（どちらも長さ round(T·fs)）"""
    n = int(round(T * fs))
    t = np.arange(n) / fs
    R = np.log(f2 / f1)
    x = np.sin(2 * np.pi * f1 * T / R * (np.exp(t * R / T) - 1.0))
    for k, sec in enumerate(fade):
        m = int(round(sec * fs))
        if m > 0:
            ramp = 0.5 - 0.5 * np.cos(np.pi * np.arange(m) / m)
            if k == 0:
                x[:m] *= ramp
            else:
                x[n - m:] *= ramp[::-1]
    v = x[::-1] * np.exp(-t * R / T)
    nfft = 1 << (2 * n - 1).bit_length()
    G = np.fft.rfft(x, nfft) * np.fft.rfft(v, nfft)
    k0 = int(round(np.sqrt(f1 * f2) * nfft / fs))
    v = v / np.abs(G[k0])
    return x, v


def ess_deconvolve(y, v, n_pre=0, n_post=None):
    """録音 y に逆フィルタ v をたたみ込み、インパルス応答を取り出す。
    時刻 0（線形のインパルス応答の始まり）は、たたみ込みの結果の len(v)−1 番目。
    n_pre 標本だけ前（負の時刻：高調波ひずみの応答が出る側）も含めて返す。
    戻り値：h（長さ n_pre + n_post）、i0（h の中で時刻 0 の位置 = n_pre）"""
    from .filters import conv_fft
    g = conv_fft(np.asarray(y, dtype=float), v)
    z = len(v) - 1
    if n_post is None:
        n_post = len(g) - z
    return g[z - n_pre:z + n_post], n_pre


def harmonic_delay(k, f1, f2, T):
    """ESS の k 次の高調波の応答が、線形の応答より前に出る時間 Δt_k = T ln k / ln(f2/f1) [s]"""
    return T * np.log(k) / np.log(f2 / f1)


def tsp(N, m=None):
    """TSP（Time-Stretched Pulse、Aoshima 1981・Suzuki ら 1995）の1周期（長さ N、2のべき乗）。
    H[k] = exp(−j4πm k²/N²)（0 ≤ k ≤ N/2）、残りは共役対称。m は「引き伸ばし」の大きさ（既定 N/4）。
    時間の信号は ifft を N/2 − m だけ巡回シフトしたもの（スイープが周期の中ほどに来る）。
    戻り値：x（長さ N、最大 1 に規格化）、H（rfft の値。逆フィルタは 1/H = conj(H)/|H|²）"""
    if m is None:
        m = N // 4
    k = np.arange(N // 2 + 1)
    H = np.exp(-1j * 4 * np.pi * m * k ** 2 / N ** 2)
    x = np.fft.irfft(H, N)
    x = np.roll(x, N // 2 - m)
    scale = np.max(np.abs(x))
    return x / scale, np.fft.rfft(x / scale)


def tsp_deconvolve(y, X, N, n_periods, skip=1):
    """TSP を n_periods 周期続けて鳴らした録音 y から、最初の skip 周期（部屋が定常になるまで）を捨てて
    残りの周期を同期加算で平均し、1周期の円状のたたみ込みを周波数の割り算で戻す。
    インパルス応答が N より短いことが前提（長いと後ろが前に回り込む）。戻り値：h（長さ N）"""
    y = np.asarray(y, dtype=float)
    segs = [y[k * N:(k + 1) * N] for k in range(skip, n_periods) if len(y) >= (k + 1) * N]
    ybar = np.mean(segs, axis=0)
    return np.fft.irfft(np.fft.rfft(ybar) / X, N)


# ---------------------------------------------------------------- マイクの指向性
def first_order_pattern(theta, p):
    """1次の指向性 G(θ) = p + (1 − p) cos θ（p = 1 無指向、0.5 単一指向（カーディオイド）、0.25 ハイパーカーディオイド、
    0 双指向（8の字））。pyroomacoustics の CardioidFamily の p と同じ"""
    return p + (1 - p) * np.cos(theta)


def directivity_factor(p):
    """指向係数 Q = 4π G(0)² / ∮ G² dΩ = 3 / (4p² − 2p + 1)（指向指数 DI = 10 log10 Q）"""
    return 3.0 / (4 * p ** 2 - 2 * p + 1)

In [ ]:
%%writefile iv2/sounds.py
"""IV-2：演習で使う音（合成音と、librosa の例の音源・自分の録音）

  x = synth_vowel(fs, sec)   声に似た合成音（基本周波数 120→180 Hz、フォルマント 700・1200・2600 Hz）
  x = pluck(fs, sec, f0)     弦をはじいた音に似た合成音（Karplus–Strong 法）
  x, name = speech(fs, wav)  wav を指定すればそれ、無ければ librosa の例 libri1（ダウンロードが必要）、
                             それもだめなら synth_vowel
"""
import numpy as np
from scipy.signal import lfilter


def synth_vowel(fs=16000, sec=2.0, f0=(120.0, 180.0), formants=((700, 80), (1200, 90), (2600, 120))):
    """声帯の音（パルス列）を、声道の共振（2次の共振器を3つ直列）に通した「あ」に近い音"""
    n = int(sec * fs)
    f = np.linspace(f0[0], f0[1], n)                       # 基本周波数をゆっくり上げる
    phase = np.cumsum(f / fs)
    src = np.diff(np.floor(phase), prepend=0.0)            # 1周期に1回だけ 1 になるパルス列
    y = src
    for fc, bw in formants:                                # 中心 fc [Hz]・帯域幅 bw [Hz] の共振器
        r = np.exp(-np.pi * bw / fs)
        a = [1.0, -2 * r * np.cos(2 * np.pi * fc / fs), r * r]
        y = lfilter([1.0 - r], a, y)
    env = np.minimum(1.0, np.minimum(np.arange(n), n - np.arange(n)) / (0.05 * fs))   # 前後 50ms で立ち上げ・下げ
    y = y * env
    return y / np.max(np.abs(y)) * 0.5


def pluck(fs=16000, sec=2.0, f0=196.0, seed=0):
    """Karplus–Strong 法：雑音を1周期分の遅延線に入れ、平均をとりながら回す（弦の減衰に似る）"""
    rng = np.random.default_rng(seed)
    n = int(sec * fs)
    P = int(round(fs / f0))
    buf = rng.uniform(-1, 1, P)
    y = np.zeros(n)
    for i in range(n):
        y[i] = buf[i % P]
        buf[i % P] = 0.996 * 0.5 * (buf[i % P] + buf[(i + 1) % P])
    return y / np.max(np.abs(y)) * 0.5


def speech(fs=16000, wav=None, sec=None):
    """話し声を返す。(x, 名前)"""
    from common.audio_io import load
    if wav:
        x, _ = load(wav, fs)
        if x.ndim == 2:
            x = x[:, 0]
        name = "own"
    else:
        try:
            import librosa
            x, _ = librosa.load(librosa.ex("libri1"), sr=fs, mono=True)
            name = "libri1"
        except Exception as e:                              # ダウンロードできないときなど
            print(f"librosa の例の音源が使えないので合成音にします（{type(e).__name__}）")
            x, name = synth_vowel(fs, 3.0), "synth_vowel"
    if sec is not None:
        x = x[:int(sec * fs)]
    return x / (np.max(np.abs(x)) + 1e-12) * 0.5, name

In [ ]:
import sys, re, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
import numpy as np
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common import plotting                      # import すると Hiragino Sans が設定される
from common import room as R
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", message="Glyph")
print("room.py を読み込めました：", len(R.__all__), "個の名前")


def listen_file(path):
    """保存した wav をそのままの大きさで再生ボタンにする。normalize=False を消さない"""
    import numpy as np, soundfile   # 各節のプログラムが sf・np などの名前を使うので、ここで読み直す
    x, fs = soundfile.read(path)
    assert np.max(np.abs(x)) <= 0.31, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが ok などの名前を使うことがあるので、確かめのセルで戻す

## 1. 共通ライブラリの照合（手順 A、`room_check.py`）

`common/room.py` の関数（音速・鏡像法・サビンとアイリング・シュレーダーの逆積分・指向性・ESS と TSP）を、pyroomacoustics と比べます。最後に「すべて一致」と出れば合格です。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-3：共通ライブラリ common/room.py を pyroomacoustics・scipy と照合する

  uv run python room_check.py

  [1] 音速：speed_of_sound と pra.parameters.calculate_speed_of_sound（湿度 0%）
  [2] 鏡像法：鏡像の位置・次数・反射の減衰、インパルス応答の全体（pra の高域通過フィルタは切って比べる）
  [3] サビン・アイリング：rt60_sabine・rt60_eyring と pra の rt60_theory、sabine_absorption と pra.inverse_sabine
  [4] シュレーダー法：decay_fit（−5〜−35 dB）と pra.experimental.measure_rt60(decay_db=30)
  [5] 指向性：first_order_pattern と pra の CardioidFamily、directivity_factor と pra の cardioid_energy
  [6] 測定信号：ESS と TSP で、分かっているインパルス応答を測り直す
最後に「すべて一致」と出れば、この章の自前の実装は信用してよい。
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import numpy as np
import pyroomacoustics as pra
from common import room as R
from common.filters import conv_fft

ok_all = True


def report(name, err, tol):
    global ok_all
    ok = bool(err <= tol)
    ok_all &= ok
    print(f"   {'一致' if ok else '不一致'}：{name}（差 {err:.2e}、許容 {tol:g}）")


print("pyroomacoustics", pra.__version__)

# ---------------------------------------------------------------- [1] 音速
print("\n[1] 音速 c(θ)：理想気体の式 331.3√(1+θ/273.15) と pra の 1次式 331.4 + 0.6θ")
for th in (0, 10, 20, 25, 30):
    c1 = float(R.speed_of_sound(th))
    c2 = pra.parameters.calculate_speed_of_sound(th, 0.0, 101.325)
    print(f"   {th:2d} ℃：自前 {c1:7.2f} m/s　pra {c2:7.2f} m/s　差 {c1 - c2:+.2f}")
th = np.linspace(0, 30, 31)
report("0〜30 ℃ の音速（2つの式は近似の仕方が違う。0.5 m/s（0.15%）まで許す）",
       np.max(np.abs(R.speed_of_sound(th) - (331.4 + 0.6 * th))), 0.5)

# ---------------------------------------------------------------- [2] 鏡像法
print("\n[2] 鏡像法：5×4×3 m、吸音率 0.3（全部の壁）、6次まで、16 kHz")
dim, src, mic, fs, alpha, N = [5.0, 4.0, 3.0], [1.2, 1.5, 1.4], [3.7, 2.6, 1.2], 16000, 0.3, 6
pra.constants.set("rir_hpf_enable", False)                 # pra の 10 Hz の高域通過フィルタを切る
room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(alpha), max_order=N, air_absorption=False)
room.add_source(src)
room.add_microphone(mic)
room.compute_rir()
s = room.sources[0]
pos, order, hits = R.image_sources(dim, src, N)
n_expect = (2 * N + 1) * (2 * N * N + 2 * N + 3) // 3
print(f"   鏡像の数：自前 {len(pos)}、pra {s.images.shape[1]}、式 (2N+1)(2N²+2N+3)/3 = {n_expect}")
key = lambda P: [tuple(v) for v in np.round(np.asarray(P, dtype=np.float64), 3)]
idx = {k: i for i, k in enumerate(key(pos))}
j = np.array([idx[k] for k in key(s.images.T)])          # pra の i 番目の鏡像 = 自前の j[i] 番目
report("鏡像の位置 [m]（pra は float32）", np.max(np.abs(pos[j] - s.images.T)), 1e-5)
report("反射の次数", float(np.max(np.abs(order[j] - s.orders))), 0)
beta = np.sqrt(1 - alpha)
report("反射による減衰 Π β^(反射の回数)", np.max(np.abs(np.prod(beta ** hits[j], axis=1) - s.damping[0])), 1e-6)
h_pra = room.rir[0][0].astype(float)
h_own, info = R.ism_rir(dim, src, mic, fs, alpha, N, pra_compat=True)
n = min(len(h_pra), len(h_own))
d = h_pra[:n] - h_own[:n]
e_db = 10 * np.log10(np.sum(d ** 2) / np.sum(h_pra ** 2))
print(f"   長さ：自前 {len(h_own)}、pra {len(h_pra)} 標本。直接音の到着 {info['t'].min() * fs:.2f} 標本"
      f"（距離 {info['dist'].min():.4f} m ÷ 343 m/s × 16000 + 遅れ 40）")
report("インパルス応答（誤差のエネルギー ÷ 全体のエネルギー、dB。−55 dB 以下を一致とする）", e_db, -55)
print(f"   最大の差 {np.abs(d).max():.2e}（直接音の山 {h_pra.max():.3f} の {np.abs(d).max() / h_pra.max() * 100:.3f}%）")

# pra の内部の sinc の表を同じように作ると、差がどこまで小さくなるか（差の原因の確かめ）
g, fdl, half = 20, 81, 40
lut_n = np.empty((fdl + 2) * g, dtype=np.float32)
acc, step = np.float32(-half - 1), np.float32(1.0) / np.float32(g)
for i in range(len(lut_n)):                                # C++ と同じく float32 の足し算で横軸を作る
    lut_n[i] = acc
    acc = np.float32(acc + step)
lut = np.where(lut_n == 0, 1.0, np.sin(np.pi * lut_n.astype(np.float32)) / (np.pi * lut_n)).astype(np.float32)
print(f"   pra の sinc の表の横軸のずれ（float32 の足し算の誤差）：最大 {np.abs(lut_n - (-41 + np.arange(len(lut_n)) / g)).max():.1e} 標本")
hann = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(fdl) / (fdl - 1))
h_emu = np.zeros(len(h_pra) + fdl)
for ti, ai in zip(info["t"].astype(np.float32), info["amp"].astype(np.float32)):
    sf = np.float32(fs) * ti
    ip = int(np.floor(sf))
    xo = np.float32(1.0 - (sf - ip)) * np.float32(g)
    lo = int(np.floor(xo))
    xf = xo - lo
    pos_l = lo + g * np.arange(fdl)
    h_emu[ip - half:ip + half + 1] += ai * hann * (lut[pos_l] + xf * (lut[pos_l + 1] - lut[pos_l]))
d2 = h_pra - h_emu[:len(h_pra)]
report("表を同じように作った場合（dB。−85 dB 以下）", 10 * np.log10(np.sum(d2 ** 2) / np.sum(h_pra ** 2)), -85)

pra.constants.set("rir_hpf_enable", True)                  # 既定に戻す
room2 = pra.ShoeBox(dim, fs=fs, materials=pra.Material(alpha), max_order=N, air_absorption=False)
room2.add_source(src)
room2.add_microphone(mic)
room2.compute_rir()
h_hpf = room2.rir[0][0].astype(float)
dh = h_hpf[:len(h_pra)] - h_pra
print(f"   参考：pra の既定（10 Hz の高域通過あり）との差は {10 * np.log10(np.sum(dh ** 2) / np.sum(h_pra ** 2)):.1f} dB"
      f"（鏡像の応答はすべて同じ符号なので直流が大きく、それが取り除かれる。100 Hz より上はほとんど同じ）")
H1, H2 = np.fft.rfft(h_pra, 8192), np.fft.rfft(h_hpf[:len(h_pra)], 8192)
fr = np.fft.rfftfreq(8192, 1 / fs)
k = fr >= 100
report("100 Hz より上の周波数応答の差（エネルギーの比 [dB]。−30 dB 以下）",
       10 * np.log10(np.sum(np.abs(H2[k] - H1[k]) ** 2) / np.sum(np.abs(H1[k]) ** 2)), -30)

# ---------------------------------------------------------------- [3] サビン・アイリング
print("\n[3] サビンとアイリングの式（6×5×3 m）")
dim3 = [6.0, 5.0, 3.0]
for a in (0.1, 0.3, 0.6):
    r3 = pra.ShoeBox(dim3, fs=fs, materials=pra.Material(a), max_order=1)
    s1, s2 = R.rt60_sabine(dim3, a), r3.rt60_theory("sabine")
    e1, e2 = R.rt60_eyring(dim3, a), r3.rt60_theory("eyring")
    print(f"   α = {a}：サビン 自前 {s1:.4f} s・pra {s2:.4f} s　アイリング 自前 {e1:.4f} s・pra {e2:.4f} s")
    report(f"α = {a} のサビンとアイリング", max(abs(s1 - s2), abs(e1 - e2)), 1e-5)
for rt in (0.15, 0.3, 0.6):
    a1, mo = pra.inverse_sabine(rt, dim3)
    report(f"inverse_sabine({rt}) の吸音率 {a1:.4f}（max_order = {mo}）", abs(R.sabine_absorption(rt, dim3) - a1), 1e-12)
# 壁ごとに吸音率が違うとき：pra の eyring は壁ごとの −S_i ln(1−α_i) の和（ミリントンの形）、自前は平均の α で計算
mat = {"east": 0.05, "west": 0.05, "north": 0.05, "south": 0.05, "floor": 0.6, "ceiling": 0.05}
r4 = pra.ShoeBox(dim3, fs=fs, materials={k: pra.Material(v) for k, v in mat.items()}, max_order=1)
al6 = [mat["west"], mat["east"], mat["south"], mat["north"], mat["floor"], mat["ceiling"]]
print(f"   床だけ α = 0.6：サビン 自前 {R.rt60_sabine(dim3, al6):.4f}・pra {r4.rt60_theory('sabine'):.4f}　"
      f"アイリング 自前（平均の α）{R.rt60_eyring(dim3, al6):.4f}・pra（壁ごと）{r4.rt60_theory('eyring'):.4f} s")
report("壁ごとに違う吸音率のサビン", abs(R.rt60_sabine(dim3, al6) - r4.rt60_theory("sabine")), 1e-5)

# ---------------------------------------------------------------- [4] シュレーダー法
print("\n[4] シュレーダーの逆積分と T30（雑音なし）")
for name, h in (("pra の部屋（6×5×3 m、RT60 0.3 s の設定）", None), ("Polack の人工応答（T60 0.5 s）", "polack")):
    if h is None:
        a, mo = pra.inverse_sabine(0.3, dim3)
        r5 = pra.ShoeBox(dim3, fs=fs, materials=pra.Material(a), max_order=mo)
        r5.add_source([1.3, 1.7, 1.4])
        r5.add_microphone([4.4, 3.8, 1.2])
        r5.compute_rir()
        h = r5.rir[0][0].astype(float)
    else:
        h = R.polack_rir(0.5, fs, 1.5, rng=0, direct=False)
    hh = h[np.argmax(np.abs(h)):]
    D = R.edc_db(R.schroeder(hh))
    t_own = R.decay_fit(D, fs, -5, -35)["T"]
    t_pra = pra.experimental.measure_rt60(hh, fs, decay_db=30)
    t_60 = pra.experimental.measure_rt60(hh, fs)
    print(f"   {name}：自前 T30 {t_own:.5f} s、pra(decay_db=30) {t_pra:.5f} s、pra の既定（decay_db=60）{t_60:.5f} s")
    report(f"{name} の T30（相対差）", abs(t_own - t_pra) / t_pra, 1e-3)

# ---------------------------------------------------------------- [5] 指向性
print("\n[5] マイクの指向性（1次の指向性 p + (1−p)cosθ）")
from pyroomacoustics.directivities import CardioidFamily
from pyroomacoustics.directivities.analytic import cardioid_energy
az = np.radians(np.arange(0, 360, 5))
for p, nm in ((1.0, "無指向"), (0.5, "単一指向（カーディオイド）"), (0.25, "ハイパーカーディオイド"), (0.0, "双指向")):
    if p < 1:
        dv = CardioidFamily(orientation=[1.0, 0.0, 0.0], p=p)
        g_pra = dv.get_response(azimuth=az, colatitude=np.full_like(az, np.pi / 2), degrees=False)
        report(f"{nm} の形", np.max(np.abs(R.first_order_pattern(az, p) - g_pra)), 1e-12)
    Q1, Q2 = R.directivity_factor(p), 4 * np.pi / cardioid_energy(p)
    print(f"   {nm}：Q = {Q1:.3f}（DI = {10 * np.log10(Q1):.2f} dB）、pra の cardioid_energy から {Q2:.3f}")
    report(f"{nm} の指向係数", abs(Q1 - Q2), 1e-12)

# ---------------------------------------------------------------- [6] ESS と TSP
print("\n[6] 測定信号：分かっているインパルス応答（Polack、T60 0.3 s、16 kHz）を測り直す")
true = R.polack_rir(0.3, fs, 0.6, rng=1)
P = 4000                                                      # 帯域を限ったデルタの前側の揺れも含めて比べる
tp = np.r_[np.zeros(P), true, np.zeros(P)]


def band_err_db(est, ref, lo, hi):
    """lo〜hi [Hz] の周波数応答の誤差（エネルギーの比、dB）"""
    E, T = np.fft.rfft(est, len(ref)), np.fft.rfft(ref)
    f = np.fft.rfftfreq(len(ref), 1 / fs)
    k = (f >= lo) & (f <= hi)
    return 10 * np.log10(np.sum(np.abs(E[k] - T[k]) ** 2) / np.sum(np.abs(T[k]) ** 2))


f1, f2, T = 50.0, 7000.0, 5.0
x, v = R.ess(f1, f2, T, fs)
h_ess, i0 = R.ess_deconvolve(conv_fft(x, true), v, n_pre=P, n_post=len(true) + P)
report("ESS（50 Hz〜7 kHz、5 秒）で測った応答の 100 Hz〜6 kHz の誤差 [dB]（−60 dB 以下）",
       band_err_db(h_ess, tp, 100, 6000), -60)
print("   注意：時刻 0 の何標本前から取り出すかで、周波数応答の誤差が変わる（帯域を限ったインパルスは時刻 0 の前にも揺れる）")
for pre in (0, 5, 40, 160):
    h_cut, _ = R.ess_deconvolve(conv_fft(x, true), v, n_pre=pre, n_post=len(true))
    est = np.r_[np.zeros(P - pre), h_cut]
    print(f"   {pre:3d} 標本（{pre / fs * 1e3:4.1f} ms）前から：100 Hz〜6 kHz {band_err_db(est, tp, 100, 6000):6.1f} dB、"
          f"100〜500 Hz {band_err_db(est, tp, 100, 500):6.1f} dB、1〜6 kHz {band_err_db(est, tp, 1000, 6000):6.1f} dB")
Nt = 1 << 15
xt, X = R.tsp(Nt)
yt = conv_fft(np.tile(xt, 4), true)[:4 * Nt]
h_tsp = R.tsp_deconvolve(yt, X, Nt, 4)
report("TSP（2^15 点、4 周期、最初の1周期を捨てて同期加算）の全帯域の誤差 [dB]（−100 dB 以下）",
       band_err_db(np.r_[np.zeros(P), h_tsp[:len(true) + P]], tp, 1, fs / 2 - 1), -100)

print("\nすべて一致" if ok_all else "\n一致しないものがあります（上の「不一致」の行を見る）")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
ok("すべて一致" in cap.stdout and "不一致" not in cap.stdout, "最後に「すべて一致」（「不一致」の行が無い）")
ok(abs(num(r"20 ℃：自前\s+([\d.]+)") - 343.21) < 0.01, "[1] 20 ℃ の音速が自前 343.21 m/s")
ok("自前 377、pra 377" in cap.stdout, "[2] 鏡像の数 377（式と一致）")
ok(num(r"インパルス応答（誤差.*?差 ([-\d.e+]+)") < -55, "[2] インパルス応答の差が −55 dB より小さい（教科書 −60.6 dB）")
ok(abs(num(r"inverse_sabine\(0\.3\) の吸音率 ([\d.]+)") - 0.3836) < 1e-4, "[3] inverse_sabine(0.3) の吸音率 0.3836")
ok(abs(num(r"自前 T30 ([\d.]+) s") - 0.3039) < 0.002, "[4] T30 が 0.304 s 前後")
ok(abs(num(r"単一指向（カーディオイド）：Q = ([\d.]+)") - 3.0) < 1e-9, "[5] カーディオイドの Q = 3.000")

## 2. 音速・音圧・FDTD・距離減衰・A 特性（IV-3-3-1〜3-3、手順 A、`sound_basics.py`）

教科書では室温を BME280（入門3 の Pico A）で読んで `--temp` に渡します。ここでは 20 ℃ にしています。自分の部屋の室温に書きかえて試してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['sound_basics.py', '--temp', '20.0']   # ノートブック版：ターミナルでの引数の代わり
"""IV-3 A：音波の基本（IV-3-3-1〜3-3）

  uv run python sound_basics.py               室温 20 ℃ として計算
  uv run python sound_basics.py --temp 26.4   BME280 などで測った室温 [℃] を使う

  [1] 音速と温度（BME280 の室温から音速を出す）
  [2] 音圧・音圧レベル・粒子速度・変位・音の強さ（20 µPa〜120 dB）
  [3] 1次元の波動方程式を、導いた2つの式（運動方程式と連続の式＋状態方程式）のまま数値で解く（FDTD）
      → パルスが c で進むこと、右へ進む波では p = ρc·u になることを確かめる
  [4] 点音源の距離減衰（pyroomacoustics の反射なしの部屋と、自前の鏡像法の 0 次）
  [5] A 特性の重み（表の値と照らし合わせる）
図：out/sound_basics.png
"""
import argparse
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
import numpy as np
import pyroomacoustics as pra
from common import plotting  # noqa: F401（日本語フォントの設定）
from common import room as R

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
ap = argparse.ArgumentParser()
ap.add_argument("--temp", type=float, default=20.0, help="室温 [℃]（BME280 で測った値など）")
a = ap.parse_args()

# ---------------------------------------------------------------- [1] 音速
GAMMA, R_GAS, M_AIR = 1.4, 8.314, 0.028965               # 比熱比、気体定数 [J/(mol K)]、空気のモル質量 [kg/mol]
print("[1] 音速 c = sqrt(γRT/M)")
for th in (0.0, 10.0, 20.0, 25.0, 30.0):
    T = th + 273.15
    print(f"   {th:4.1f} ℃：sqrt(γRT/M) = {np.sqrt(GAMMA * R_GAS * T / M_AIR):6.2f} m/s　"
          f"331.3√(1+θ/273.15) = {float(R.speed_of_sound(th)):6.2f}　1次の近似 331.3+0.606θ = {331.3 + 0.606 * th:6.2f}")
c = float(R.speed_of_sound(a.temp))
print(f"   室温 {a.temp:.1f} ℃ の音速：{c:.2f} m/s（20 ℃ の 343.21 m/s との違い {100 * (c / 343.21 - 1):+.2f}%）")
print(f"   1 ℃ あたりの変化 dc/dθ = {float(R.speed_of_sound(a.temp + 0.5) - R.speed_of_sound(a.temp - 0.5)):.3f} m/s。"
      f"3 m を進む時間は {3 / c * 1e3:.3f} ms（16 kHz で {3 / c * 16000:.1f} 標本）")

# ---------------------------------------------------------------- [2] 音圧と粒子速度
P0 = 101325.0                                             # 大気圧 [Pa]
rho = P0 * M_AIR / (R_GAS * (a.temp + 273.15))            # 空気の密度（理想気体）[kg/m³]
Z0 = rho * c                                              # 特性インピーダンス [Pa·s/m]
print(f"\n[2] 空気の密度 ρ = {rho:.3f} kg/m³、特性インピーダンス ρc = {Z0:.1f} Pa·s/m（{a.temp:.1f} ℃、1 気圧）")
print("   音圧レベル | 音圧の実効値 | 粒子速度の実効値 | 1 kHz の変位の振幅 | 音の強さ | 強さのレベル")
for Lp in (0, 30, 60, 94, 120):
    p = R.P_REF * 10 ** (Lp / 20)
    u = p / Z0
    xi = np.sqrt(2) * u / (2 * np.pi * 1000)              # 振幅 = √2 × 実効値、変位 = 速度 / ω
    I = p ** 2 / Z0
    print(f"   {Lp:5.0f} dB | {p:10.3e} Pa | {u:10.3e} m/s | {xi:10.3e} m | {I:9.2e} W/m² | {10 * np.log10(I / 1e-12):6.2f} dB")
print(f"   1 Pa は {float(R.spl_db(1.0)):.2f} dB SPL（校正器の 94 dB）。強さのレベルと音圧レベルの差は "
      f"10log10(p_ref²/(ρc·10⁻¹²)) = {10 * np.log10(R.P_REF ** 2 / (Z0 * 1e-12)):+.2f} dB")

# ---------------------------------------------------------------- [3] 1次元の FDTD
print("\n[3] 1次元の波動方程式を、運動方程式 ρ∂u/∂t = −∂p/∂x と、連続の式＋状態方程式 ∂p/∂t = −ρc²∂u/∂x で解く")
dx = 0.01                                                 # 1 cm
dt = 0.5 * dx / c                                         # クーラン数 0.5（安定の条件 c·dt/dx ≤ 1）
x = np.arange(0, 10.0, dx)                                # 10 m の管（両端は閉じた壁：u = 0）
p = np.exp(-((x - 5.0) / 0.1) ** 2)                       # はじめ：真ん中に圧力の山、空気は止まっている
u = np.zeros(len(x) + 1)                                  # u は p の間（半分ずらした格子）に置く
snaps = {0.0: p.copy()}
n_steps = int(round(0.008 / dt))                          # 8 ms 進める
for n in range(1, n_steps + 1):
    u[1:-1] -= dt / (rho * dx) * (p[1:] - p[:-1])         # 運動方程式
    p -= dt * rho * c ** 2 / dx * (u[1:] - u[:-1])        # 連続の式＋状態方程式
    if n in (n_steps // 2, n_steps):
        snaps[n * dt] = p.copy()
t_end = n_steps * dt
right = x > 5.0
i_pk = np.argmax(np.where(right, p, -np.inf))
c_est = (x[i_pk] - 5.0) / t_end
u_mid = 0.5 * (u[1:] + u[:-1])                            # p と同じ位置に直した u
print(f"   {t_end * 1e3:.2f} ms 後、右へ進む山の位置 {x[i_pk]:.3f} m → 速さ {c_est:.1f} m/s（c = {c:.1f}、格子 1 cm の分だけずれうる）")
print(f"   山の高さ {p[i_pk]:.3f}（はじめの山 1 の半分ずつが左右に分かれる：ダランベールの解）")
print(f"   右へ進む山で p/u = {p[i_pk] / u_mid[i_pk]:.1f} Pa·s/m（ρc = {Z0:.1f}）。左へ進む山では "
      f"{p[np.argmax(np.where(~right, p, -np.inf))] / u_mid[np.argmax(np.where(~right, p, -np.inf))]:.1f}（符号が逆）")

# ---------------------------------------------------------------- [4] 距離減衰
print("\n[4] 点音源の距離減衰（反射のない部屋、16 kHz）")
fs = 16000
dists = np.array([0.25, 0.5, 1.0, 2.0, 4.0])
e_pra, e_own = [], []
for r in dists:
    room = pra.AnechoicRoom(fs=fs)
    room.add_source([0.0, 0.0, 0.0])
    room.add_microphone([r, 0.0, 0.0])
    room.compute_rir()
    e_pra.append(np.sum(room.rir[0][0].astype(float) ** 2))
    h, _ = R.ism_rir([100.0, 100.0, 100.0], [50.0, 50.0, 50.0], [50.0 + r, 50.0, 50.0], fs, 1.0, 0)
    e_own.append(np.sum(h ** 2))
L_pra = 10 * np.log10(np.array(e_pra) / e_pra[2])
L_own = 10 * np.log10(np.array(e_own) / e_own[2])
for r, l1, l2 in zip(dists, L_pra, L_own):
    print(f"   r = {r:4.2f} m：1 m を基準に pra {l1:+6.2f} dB、自前 {l2:+6.2f} dB、式 −20log10(r) = {-20 * np.log10(r):+6.2f} dB")
print(f"   距離が2倍になるごとに {-np.mean(np.diff(L_pra)):.2f} dB 下がる（式は 20log10 2 = 6.02 dB）")
print(f"   1 m での大きさの比（pra ÷ 自前）= {np.sqrt(e_pra[2] / e_own[2]):.4f}（4π = {4 * np.pi:.4f}：pra は 1/(4π) を掛けない）")

# ---------------------------------------------------------------- [5] A 特性
print("\n[5] A 特性の重み [dB]（IEC 61672-1 の式。右は規格の表として広く引用される値）")
table = {63: -26.2, 125: -16.1, 250: -8.6, 500: -3.2, 1000: 0.0, 2000: 1.2, 4000: 1.0, 8000: -1.1}
for f, v in table.items():
    print(f"   {f:5d} Hz：式 {float(R.a_weighting_db(f)):+6.2f}　表 {v:+5.1f}")
print(f"   最大の差 {max(abs(float(R.a_weighting_db(f)) - v) for f, v in table.items()):.2f} dB（表は 0.1 dB に丸めてある）")

# ---------------------------------------------------------------- 図
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for t, s in snaps.items():
    ax[0].plot(x, s, label=f"{t * 1e3:.1f} ms")
ax[0].set_xlabel("位置 x [m]")
ax[0].set_ylabel("音圧（はじめの山を 1）")
ax[0].set_title("1次元の FDTD：山が左右に c で分かれて進む")
ax[0].legend()
rr = np.logspace(np.log10(0.2), np.log10(5), 100)
ax[1].semilogx(rr, -20 * np.log10(rr), "k-", label="−20log10 r（逆2乗則）")
ax[1].semilogx(dists, L_pra, "o", label="pyroomacoustics（反射なし）")
ax[1].set_xlabel("距離 r [m]")
ax[1].set_ylabel("1 m を基準にしたレベル [dB]")
ax[1].set_title("点音源の距離減衰（2倍で −6 dB）")
ax[1].grid(True, which="both", alpha=0.3)
ax[1].legend()
fig.tight_layout()
fig.savefig(OUT / "sound_basics.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print("\n図：out/sound_basics.png")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
ok(abs(num(r"室温 20\.0 ℃ の音速：([\d.]+)") - 343.21) < 0.01, "[1] 20 ℃ の音速 343.21 m/s")
ok(abs(num(r"速さ ([\d.]+) m/s") - 342.6) < 1.0, "[3] FDTD の山の速さ 342.6 m/s 前後")
ok(abs(num(r"山の高さ ([\d.]+)") - 0.5) < 0.01, "[3] 山の高さ 0.499（半分ずつ左右に分かれる）")
ok(abs(num(r"p/u = ([\d.]+)") - 414.1) < 2, "[3] p/u ≈ ρc（414.1）")
ok(abs(num(r"= ([\d.]+)（4π") - 4 * np.pi) < 0.05, "[4] pra ÷ 自前 = 12.56（4π）")

## 3. インパルス応答の中身を見て、聞く（IV-3-3-4・3-6、手順 B、`rir_anatomy.py`）

反射の密度、DRR と臨界距離、直接音／初期反射／後部残響への分け方を確かめ、それぞれを話し声にたたみ込んで聞きます（10〜20 秒）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-3 B：部屋のインパルス応答の中身（IV-3-3-4・3-6）

  uv run python rir_anatomy.py

  [1] 反射の密度：時刻 t までに届く鏡像の数を数え、式 4πc³t²/V と比べる
  [2] 直接音と残響音のエネルギー比（DRR）と臨界距離：距離を変えて、式 (r_c / r)² と比べる
  [3] 直接音・初期反射（50 ms まで）・後部残響に分け、C50 を求め、声にたたみ込んで聞く
  [4] 同じ部屋で、マイクを近く（0.5 m）と遠く（2.5 m）に置いた声を聞き比べる
wav：out/anat_*.wav（すべて同じ倍率で保存。最大 0.3 以下）  図：out/rir_anatomy.png
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
import numpy as np
import pyroomacoustics as pra
from common import plotting  # noqa: F401
from common import room as R
from common.audio_io import save
from common.filters import conv_fft
from iv2.sounds import speech

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
FS, C = 16000, 343.0
DIM = [6.0, 5.0, 3.0]
V, S = R.room_volume_surface(DIM)


def rir(rt60, src, mic):
    alpha, N = pra.inverse_sabine(rt60, DIM)
    room = pra.ShoeBox(DIM, fs=FS, materials=pra.Material(alpha), max_order=N, air_absorption=False)
    room.add_source(src)
    room.add_microphone(mic)
    room.compute_rir()
    return room.rir[0][0].astype(float), alpha


# ---------------------------------------------------------------- [1] 反射の密度
print("[1] 反射の密度（6×5×3 m、音源 (1.3, 1.7, 1.4)、マイク (4.4, 3.8, 1.2)）")
src, mic = np.array([1.3, 1.7, 1.4]), np.array([4.4, 3.8, 1.2])
N = 20
pos, order, _ = R.image_sources(DIM, src, N)
t_arr = np.linalg.norm(pos - mic, axis=1) / C
t_ok = N / np.sqrt(np.sum(1 / np.asarray(DIM) ** 2)) / C   # 次数 N の正八面体に入る球の半径 / c：この時刻までは全部の鏡像がある
print(f"   次数 {N} までの鏡像 {len(pos)} 個。{t_ok * 1e3:.0f} ms までは、届く鏡像がすべてそろっている")
print("   時刻 [ms] | それまでに届いた数 | 式 4πc³t³/(3V) | 1 ms あたりの密度（数えた値・式 4πc³t²/V）")
for t in (0.010, 0.020, 0.050, 0.080, 0.120):
    n_count = np.sum(t_arr <= t)
    n_th = 4 * np.pi * C ** 3 * t ** 3 / (3 * V)
    dens = np.sum((t_arr > t - 0.0025) & (t_arr <= t + 0.0025)) / 5.0
    print(f"   {t * 1e3:6.0f} | {n_count:6d} | {n_th:8.1f} | {dens:7.1f}・{4 * np.pi * C ** 3 * t ** 2 / V / 1000:7.1f}")
t_mix = np.sqrt(V / (4 * np.pi * C ** 3) * 1000)       # 密度が 1 ms に 1 個になる時刻
print(f"   密度が 1 ms あたり 1 個を超える時刻：{t_mix * 1e3:.1f} ms（ここから先は1つ1つの反射を聞き分けにくい）")

# ---------------------------------------------------------------- [2] DRR と臨界距離
print("\n[2] DRR と臨界距離（設定 0.5 s の部屋。音源は (1.7, 2.1, 1.35)、マイクを斜めの方向 u に離す）")
src2 = np.array([1.7, 2.1, 1.35])
U = np.array([0.85, 0.5, 0.16]) / np.linalg.norm([0.85, 0.5, 0.16])
h_ref, alpha = rir(0.5, src2, src2 + 1.0 * U)
T30 = R.rt60_from_ir(h_ref, FS, "T30", noise="none")["T"]
rc_sab = R.critical_distance(DIM, 0.5)
rc_meas = R.critical_distance(DIM, T30)
print(f"   臨界距離：サビン（0.5 s）から {rc_sab:.3f} m、測った T30（{T30:.3f} s）から {rc_meas:.3f} m")
dists = [0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5]
drr = []
for r in dists:
    h, _ = rir(0.5, src2, src2 + r * U)
    drr.append(R.drr_db(h, FS))
    print(f"   r = {r:4.2f} m：DRR {drr[-1]:+6.1f} dB　式 20log10(r_c/r) = {20 * np.log10(rc_sab / r):+6.1f}（サビン）・"
          f"{20 * np.log10(rc_meas / r):+6.1f}（T30）")
slope = np.polyfit(np.log10(dists), drr, 1)[0]
print(f"   DRR の距離についての傾き {slope:.1f} dB/桁（式は −20 dB/桁 ＝ 2倍で −6 dB）")
# 落とし穴：音源とマイクを部屋の真ん中の面（y = Ly/2、z = Lz/2）にそろえると、鏡像が同じ時刻に重なる。
# 同じ x の位置で、y・z だけを変えて比べる（マイクは音源から +x に r）
print("   （落とし穴）音源とマイクを部屋の真ん中の面にそろえた場合（マイクは +x 方向。式は "
      f"0.3 m で {20 * np.log10(rc_sab / 0.3):+.1f}、1.0 m で {20 * np.log10(rc_sab / 1.0):+.1f} dB）")
for nm, c0 in (("y・z とも真ん中 (2.0, 2.5, 1.5)", [2.0, 2.5, 1.5]), ("z だけ真ん中 (2.0, 2.2, 1.5)", [2.0, 2.2, 1.5]),
               ("y は真ん中、z を 2 cm ずらす (2.0, 2.5, 1.52)", [2.0, 2.5, 1.52]),
               ("y・z ともずらす (2.0, 2.2, 1.3)", [2.0, 2.2, 1.3])):
    c0 = np.array(c0)
    vals = []
    for r in (0.3, 1.0):
        h_sym, _ = rir(0.5, c0, c0 + [r, 0, 0])
        vals.append(f"{r} m で DRR {R.drr_db(h_sym, FS):+.1f} dB・T30 {R.rt60_from_ir(h_sym, FS, 'T30', noise='none')['T']:.3f} s")
    print(f"     {nm}：" + "、".join(vals))
print("   真ん中の面では上下（左右）の鏡像がマイクから同じ距離になり、振幅で足し合わさって残響のエネルギーが増える")

# ---------------------------------------------------------------- [3] 直接音・初期反射・残響
print("\n[3] インパルス応答を3つに分ける（設定 0.3 s と 0.8 s、マイクは音源から 1.5 m）")
x, name = speech(FS, sec=4.0)
print(f"   音源：{name}（{len(x) / FS:.1f} 秒）")
parts_for_plot = None
store = {}
for rt in (0.3, 0.8):
    h, _ = rir(rt, src2, src2 + 1.5 * U)
    i = int(np.argmax(np.abs(h)))
    w = int(0.0025 * FS)
    k50 = i + int(0.050 * FS)
    direct = np.zeros_like(h)
    direct[:i + w + 1] = h[:i + w + 1]
    early = np.zeros_like(h)
    early[:k50] = h[:k50]
    late = h - early
    e = h ** 2
    print(f"   設定 {rt} s：直接音 {10 * np.log10(e[:i + w + 1].sum() / e.sum()):+.1f} dB、"
          f"50 ms までの反射 {10 * np.log10(e[i + w + 1:k50].sum() / e.sum()):+.1f} dB、"
          f"後部残響 {10 * np.log10(e[k50:].sum() / e.sum()):+.1f} dB（全体を 0 dB）。"
          f"C50 = {R.clarity_db(h, FS):+.1f} dB、DRR = {R.drr_db(h, FS):+.1f} dB")
    for nm, hh in (("direct", direct), ("early", early), ("full", h), ("late", late)):
        store[f"anat_rt{int(rt * 1000)}_{nm}"] = conv_fft(x, hh)[:len(x) + int(0.5 * FS)]
    if rt == 0.3:
        parts_for_plot = (h, i, w, k50)

# ---------------------------------------------------------------- [4] 近くと遠く
print("\n[4] 近く（0.5 m）と遠く（2.5 m）。設定 0.5 s")
for r in (0.5, 2.5):
    h, _ = rir(0.5, src2, src2 + r * U)
    y = conv_fft(x, h)[:len(x) + int(0.5 * FS)]
    store[f"anat_dist{int(r * 100):03d}cm"] = y
    print(f"   r = {r} m：DRR {R.drr_db(h, FS):+.1f} dB")
e05 = np.sum(store["anat_dist050cm"] ** 2)
e30 = np.sum(store["anat_dist250cm"] ** 2)
print(f"   2.5 m は 0.5 m より {10 * np.log10(e30 / e05):+.1f} dB（直接音だけなら 20log10(0.5/2.5) = {20 * np.log10(0.5 / 2.5):+.1f} dB。"
      "残響は距離でほとんど変わらないので、思ったほど小さくならない）")

g = 0.3 / max(np.max(np.abs(v)) for v in store.values())   # 全部を同じ倍率にする（大小の比較ができるように）
save(OUT / "anat_dry.wav", x * g, FS)
for k, v in store.items():
    save(OUT / f"{k}.wav", v * g, FS)
print(f"\nwav：out/anat_dry.wav と {len(store)} 個（同じ倍率 {g:.3g}）。direct → early → full → late の順に聞く")

# ---------------------------------------------------------------- 図
h, i, w, k50 = parts_for_plot
t = np.arange(len(h)) / FS * 1e3
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(t, h, color="0.6", lw=0.6, label="全体")
ax[0].plot(t[:i + w + 1], h[:i + w + 1], color="C3", label="直接音")
ax[0].plot(t[i + w + 1:k50], h[i + w + 1:k50], color="C0", lw=0.8, label="初期反射（50 ms まで）")
ax[0].set_xlim(0, 300)
ax[0].set_xlabel("時間 [ms]")
ax[0].set_title("インパルス応答（設定 0.3 s、1.5 m）")
ax[0].legend()
tt = np.linspace(0.001, 0.15, 200)
ax[1].plot(tt * 1e3, [np.sum(t_arr <= v) for v in tt], label="数えた鏡像の数")
ax[1].plot(tt * 1e3, 4 * np.pi * C ** 3 * tt ** 3 / (3 * V), "--", label="式 4πc³t³/(3V)")
ax[1].set_xlabel("時間 [ms]")
ax[1].set_ylabel("それまでに届いた反射の数")
ax[1].set_title("反射は t² に比例して密になる")
ax[1].legend()
rr = np.logspace(np.log10(0.25), np.log10(4), 50)
ax[2].semilogx(dists, drr, "o-", label="鏡像法の DRR")
ax[2].semilogx(rr, 20 * np.log10(rc_sab / rr), "--", label=f"式（サビン、r_c = {rc_sab:.2f} m）")
ax[2].axhline(0, color="k", lw=0.5)
ax[2].set_xlabel("音源からの距離 [m]")
ax[2].set_ylabel("DRR [dB]")
ax[2].set_title("DRR と臨界距離（DRR = 0 dB の距離）")
ax[2].legend()
fig.tight_layout()
fig.savefig(OUT / "rir_anatomy.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print("図：out/rir_anatomy.png")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
ok(abs(num(r"50 \|\s+(\d+) \|") - 229) <= 5, "[1] 50 ms までに数えた反射 229 個前後（式 234.8）")
ok(abs(num(r"1 個を超える時刻：([\d.]+) ms") - 13.3) < 0.5, "[1] 密度が 1 ms に 1 個を超えるのは 13.3 ms")
ok(abs(num(r"サビン（0\.5 s）から ([\d.]+) m") - 0.760) < 0.005, "[2] 臨界距離 0.760 m（サビン）")
ok(abs(num(r"傾き ([-\d.]+) dB/桁") + 20) < 1.5, "[2] DRR の傾き −20 dB/桁前後（教科書 −20.1）")
ok(abs(num(r"設定 0\.3 s：.*?C50 = ([-+\d.]+) dB") - 13.0) < 1.0, "[3] 設定 0.3 s で C50 = +13.0 dB 前後")
ok(abs(num(r"設定 0\.8 s：.*?C50 = ([-+\d.]+) dB") - 2.5) < 1.0, "[3] 設定 0.8 s で C50 = +2.5 dB 前後")

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['anat_dry.wav', 'anat_rt300_direct.wav', 'anat_rt300_early.wav', 'anat_rt300_full.wav', 'anat_rt300_late.wav', 'anat_rt800_full.wav', 'anat_dist050cm.wav', 'anat_dist250cm.wav']:
    print(f)
    listen_file("out/" + f)

## 4. サビン・アイリングと鏡像法（IV-3-3-5・3-8、手順 C、`sabine_ism.py`）

自分の部屋のサビン・アイリングの予測は、先にノートに手で計算しておいてください（教科書の手順 C の 6）。鏡像法の計算に 20 秒〜1 分かかります。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-3 C：サビン・アイリングの式と、鏡像法の部屋の残響時間を比べる（IV-3-3-5・3-8）

  uv run python sabine_ism.py

  [1] pra.inverse_sabine で作った部屋（6×5×3 m。第7章・IV-5 と同じ作り方）の残響時間を、
      サビン・アイリングの予測、シュレーダー法の T20・T30、pra.measure_rt60 の既定（decay_db=60）で比べる
  [2] 鏡像法の減衰を「方向ごとの反射の回数」から予測する（本文の式）。T30 がアイリングより長くなる理由
  [3] 反射の次数 max_order を減らすと、減衰曲線の後ろが切れる
  [4] 床だけよく吸う部屋（拡散音場の仮定が崩れる）
図：out/sabine_ism.png
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
import numpy as np
import pyroomacoustics as pra
from common import plotting  # noqa: F401
from common import room as R

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
FS, C = 16000, 343.0
DIM = [6.0, 5.0, 3.0]
SRC, MIC = [1.3, 1.7, 1.4], [4.4, 3.8, 1.2]


def simulate(dim, materials, max_order):
    room = pra.ShoeBox(dim, fs=FS, materials=materials, max_order=max_order, air_absorption=False)
    room.add_source(SRC)
    room.add_microphone(MIC)
    room.compute_rir()
    return room.rir[0][0].astype(float)


def edc_of(h):
    h = h[np.argmax(np.abs(h)):]
    return R.edc_db(R.schroeder(h))


def direction_average_edc(dim, alpha, tmax, n_dir=20000, seed=0):
    """鏡像法の減衰の予測：方向 u に進む音は、1 m あたり Σ|u_d|/L_d 回反射する。
    エネルギーの包絡 ∝ ⟨(1−α)^(c t Σ|u_d|/L_d)⟩（球面の上の平均）。それを後ろから積分して EDC にする"""
    rng = np.random.default_rng(seed)
    u = rng.standard_normal((n_dir, 3))
    u /= np.linalg.norm(u, axis=1, keepdims=True)
    rate = (np.abs(u) / np.asarray(dim, dtype=float)).sum(axis=1)      # [1/m]
    t = np.arange(0, tmax, 1 / FS)
    env = np.exp(np.outer(t, C * rate * np.log(1 - alpha))).mean(axis=1)
    return R.edc_db(np.cumsum(env[::-1])[::-1]), rate


# ---------------------------------------------------------------- [1]・[2]
print("[1] 6×5×3 m、pra.inverse_sabine で吸音率と次数を決めた部屋")
print("   設定 | 吸音率 | 次数 | サビン | アイリング | T20 | T30 | pra 既定(60 dB) | 方向平均の予測 T30")
curves = {}
for rt in (0.15, 0.3, 0.5, 0.8):
    alpha, N = pra.inverse_sabine(rt, DIM)
    h = simulate(DIM, pra.Material(alpha), N)
    T20 = R.rt60_from_ir(h, FS, "T20", noise="none")["T"]
    T30 = R.rt60_from_ir(h, FS, "T30", noise="none")["T"]
    m60 = pra.experimental.measure_rt60(h, FS)
    D_th, rate = direction_average_edc(DIM, alpha, 3 * rt)
    T_th = R.decay_fit(D_th, FS)["T"]
    curves[rt] = (edc_of(h), D_th)
    print(f"   {rt:4.2f} s | {alpha:.3f} | {N:3d} | {R.rt60_sabine(DIM, alpha):.3f} | {R.rt60_eyring(DIM, alpha):.3f} | "
          f"{T20:.3f} | {T30:.3f} | {m60:.3f} | {T_th:.3f}")
S_over_4V = room_S = R.room_volume_surface(DIM)[1] / (4 * np.prod(DIM))
print(f"\n[2] 1 m あたりの反射の回数 Σ|u_d|/L_d の、方向についての平均 {rate.mean():.4f} /m"
      f"（式 S/(4V) = {S_over_4V:.4f}）")
L_ = np.asarray(DIM)
print(f"   厳密な最小 1/max L = {1 / L_.max():.4f}（長い辺に沿う向き）、最大 √(Σ1/L²) = {np.sqrt(np.sum(1 / L_ ** 2)):.4f}"
      f"（乱数で選んだ {len(rate)} 方向では {rate.min():.4f}〜{rate.max():.4f}）")
print("   平均の回数で減るとみなすとアイリングの式。実際は方向ごとに違う速さの指数関数の平均なので、")
print("   遅く減る方向（長い辺に沿って進む音）が後ろで残り、T30 はアイリングより長くなる（イェンセンの不等式）")

# ---------------------------------------------------------------- [3] 次数
print("\n[3] 設定 0.5 s の部屋で、反射の次数を変える")
alpha, N = pra.inverse_sabine(0.5, DIM)
order_curves = {}
for n_ord in (N // 4, N // 2, N):
    h = simulate(DIM, pra.Material(alpha), n_ord)
    D = edc_of(h)
    order_curves[n_ord] = D
    t_cut = n_ord / np.sqrt(np.sum(1 / np.asarray(DIM) ** 2)) / C    # 3次元の正八面体に入る球の半径 / c
    T30 = R.decay_fit(D, FS)["T"]
    print(f"   次数 {n_ord:3d}：長さ {len(h) / FS:.2f} s、T30 {T30:.3f} s、"
          f"全方向の鏡像がそろっている時間の目安 {t_cut:.3f} s（それより後ろは鏡像が欠けて急に落ちる）")

# ---------------------------------------------------------------- [4] 床だけ吸う部屋
print("\n[4] 床だけ α = 0.6、ほかの壁は 0.15（厚いカーペットの部屋のような、拡散でない部屋）")
mat = {"east": 0.15, "west": 0.15, "north": 0.15, "south": 0.15, "floor": 0.6, "ceiling": 0.15}
al6 = [mat["west"], mat["east"], mat["south"], mat["north"], mat["floor"], mat["ceiling"]]
h_u = simulate(DIM, {k: pra.Material(v) for k, v in mat.items()}, 90)   # 次数 60・90・120 で T30 が変わらないことを確かめた
room_u = pra.ShoeBox(DIM, fs=FS, materials={k: pra.Material(v) for k, v in mat.items()}, max_order=1)
T20u = R.rt60_from_ir(h_u, FS, "T20", noise="none")["T"]
T30u = R.rt60_from_ir(h_u, FS, "T30", noise="none")["T"]
EDTu = R.rt60_from_ir(h_u, FS, "EDT", noise="none")["T"]
print(f"   サビン {R.rt60_sabine(DIM, al6):.3f} s、アイリング（平均の α）{R.rt60_eyring(DIM, al6):.3f} s、"
      f"ミリントン型（pra の rt60_theory('eyring')）{room_u.rt60_theory('eyring'):.3f} s")
print(f"   鏡像法（次数 90）：EDT {EDTu:.3f} s、T20 {T20u:.3f} s、T30 {T30u:.3f} s"
      "（水平に進む音は床に当たらないので吸われにくく、後ろの減衰が遅い。どの式も「拡散音場」を仮定している）")
D_u = edc_of(h_u)

# ---------------------------------------------------------------- 図
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, (rt, (D, Dth)) in enumerate(curves.items()):
    t = np.arange(len(D)) / FS
    ax[0].plot(t, D, color=f"C{i}", label=f"設定 {rt} s（鏡像法）")
    ax[0].plot(np.arange(len(Dth)) / FS, Dth, "--", color=f"C{i}", lw=1)
    ax[0].plot([0, rt], [0, -60], ":", color=f"C{i}", lw=1)
ax[0].set_ylim(-80, 2)
ax[0].set_xlim(0, 1.6)
ax[0].set_xlabel("時間 [s]")
ax[0].set_ylabel("EDC [dB]")
ax[0].set_title("実線：鏡像法、破線：方向平均の予測、点線：サビン")
ax[0].legend(fontsize=8)
for n_ord, D in order_curves.items():
    ax[1].plot(np.arange(len(D)) / FS, D, label=f"次数 {n_ord}")
ax[1].set_ylim(-80, 2)
ax[1].set_xlabel("時間 [s]")
ax[1].set_title("反射の次数と EDC（設定 0.5 s）")
ax[1].legend()
ax[2].plot(np.arange(len(D_u)) / FS, D_u, label="床だけ α = 0.6（鏡像法）")
ax[2].plot([0, R.rt60_sabine(DIM, al6)], [0, -60], ":", label="サビン")
ax[2].plot([0, room_u.rt60_theory("eyring")], [0, -60], "--", label="ミリントン型")
ax[2].set_ylim(-80, 2)
ax[2].set_xlabel("時間 [s]")
ax[2].set_title("拡散でない部屋：EDC が折れ曲がる")
ax[2].legend()
fig.tight_layout()
fig.savefig(OUT / "sabine_ism.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print("\n図：out/sabine_ism.png")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
row = re.search(r"0\.50 s \| ([\d.]+) \|\s+(\d+) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+) \| ([\d.]+)", cap.stdout)
t20, t30, t60pra, pred = (float(row.group(i)) for i in (5, 6, 7, 8))
ok(abs(t20 - 0.512) < 0.01 and abs(t30 - 0.561) < 0.01, f"[1] 設定 0.5 s で T20 {t20}・T30 {t30} s（教科書 0.512・0.561）")
ok(abs(t60pra - 0.678) < 0.01 and abs(pred - 0.548) < 0.01, f"[1] pra 既定 {t60pra} s・方向平均の予測 {pred} s（教科書 0.678・0.548）")
ok(abs(num(r"方向についての平均 ([\d.]+) /m") - 0.3507) < 0.003, "[2] 反射の回数の方向平均 0.3507 /m（S/(4V) = 0.3500）")
ok(abs(num(r"T30 ([\d.]+) s（水平") - 0.913) < 0.03, "[4] 床だけ吸う部屋の T30 0.913 s 前後（サビン 0.448 s よりずっと長い）")

## 5. シュレーダー法と雑音の床、T60 の不確かさ（IV-3-3-7・3-12、手順 D、`noise_floor.py`）

図の左で、そのまま積分した EDC（赤）が −25 dB あたりから平らになり、打ち切り＋補正（青）が真の直線（黒の破線）に沿っていることを確かめてください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-3 D：シュレーダー法と雑音の床、T60 の推定の不確かさ（IV-3-3-7・3-12）

  uv run python noise_floor.py

  答えの分かっている人工のインパルス応答（Polack のモデル：白色雑音 × 指数の減衰、T60 = 0.5 s）に
  測定の雑音を足し、3つの方法で T20・T30 を求める。
  [1] 雑音の床の高さ（PNR）と、3つの方法（そのまま／雑音を引く（Chu）／打ち切り＋補正（Lundeby））のかたより
  [2] 統計の誤用：EDC への直線の当てはめの「標準誤差」を、T60 の不確かさとして報告する
  [3] 帯域ごとの T30 のばらつき（狭い帯域ほど、1本の応答から求めた値がばらつく）
図：out/noise_floor.png
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from common import plotting  # noqa: F401
from common import room as R

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
FS, T60, SEC, N_SEED = 16000, 0.5, 2.0, 30

# ---------------------------------------------------------------- [1]
print(f"[1] 真の T60 = {T60} s、長さ {SEC} s、{N_SEED} 本ずつ。平均 ± 標準偏差 [s]（測れなかった本数）")
print("   PNR は「残響の始まりのパワー」と「雑音のパワー」の比。—— は全部の本で −25・−35 dB まで下がらず測れない")
methods = {"none": "そのまま", "chu": "雑音を引く", "lundeby": "打ち切り＋補正"}
for pnr in (80, 60, 45, 35, 25):
    line = []
    for m in methods:
        for meth in ("T20", "T30"):
            v = np.array([R.rt60_from_ir(R.polack_rir(T60, FS, SEC, rng=s, direct=False, pnr_db=pnr),
                                         FS, meth, noise=m)["T"] for s in range(N_SEED)])
            ok = np.isfinite(v)
            line.append(f"{meth} {np.mean(v[ok]):5.3f}±{np.std(v[ok], ddof=1):5.3f}" + (f"（{np.sum(~ok)}）" if np.sum(~ok) else "")
                        if ok.sum() > 1 else f"{meth}   ——   ")
    print(f"   PNR {pnr:2d} dB：" + " ｜ ".join(f"{methods[m]} " + "、".join(line[2 * i:2 * i + 2]) for i, m in enumerate(methods)))

# ---------------------------------------------------------------- [2] 統計の誤用
print("\n[2] 当てはめの標準誤差と、本当のばらつき（PNR 70 dB、T30、Lundeby、100 本）")
T, SE = [], []
for s in range(100):
    h = R.polack_rir(T60, FS, 1.5, rng=s, direct=False, pnr_db=70)
    r = R.rt60_from_ir(h, FS, "T30", noise="lundeby")
    f = r["fit"]
    t = np.arange(f["i0"], f["i1"]) / FS
    ols = sm.OLS(r["D"][f["i0"]:f["i1"]], sm.add_constant(t)).fit()
    b, se_b = ols.params[1], ols.bse[1]
    T.append(-60 / b)
    SE.append(60 / b ** 2 * se_b)                         # デルタ法：T = −60/b の標準誤差
T, SE = np.array(T), np.array(SE)
print(f"   T30 の平均 {T.mean():.4f} s、100 本の間の標準偏差（本当のばらつき）{T.std(ddof=1):.4f} s")
print(f"   1本ごとの当てはめの標準誤差（OLS、点の数 {len(t)}）の中央値 {np.median(SE):.5f} s"
      f" → 本当のばらつきの 1/{T.std(ddof=1) / np.median(SE):.0f}")
print("   EDC は後ろからの累積和なので、隣どうしの点がほとんど同じ値（強い自己相関）。")
print("   点を独立とみなす OLS の標準誤差は、ばらつきをひどく小さく見積もる（95% 区間を出してはいけない）")
cover = np.mean(np.abs(T - T60) <= 1.96 * SE)
print(f"   OLS の 95% 区間が真の T60 を含んだ割合：{cover * 100:.0f}%（95% のはず）")

# ---------------------------------------------------------------- [3] 帯域ごと
print("\n[3] 1オクターブの帯域ごとの T30（PNR 70 dB、Lundeby、100 本）")
bands = (250, 1000, 4000)
res = {fc: [] for fc in bands}
for s in range(100):
    h = R.polack_rir(T60, FS, 1.5, rng=1000 + s, direct=False, pnr_db=70)
    for fc in bands:
        res[fc].append(R.rt60_from_ir(R.octave_band(h, FS, fc), FS, "T30", noise="lundeby")["T"])
for fc in bands:
    v = np.array(res[fc])
    print(f"   {fc:5d} Hz：平均 {np.nanmean(v):.3f} s、標準偏差 {np.nanstd(v, ddof=1):.4f} s"
          f"（帯域幅 {fc / np.sqrt(2) * (2 - 1):.0f} Hz・変動係数 {np.nanstd(v, ddof=1) / np.nanmean(v) * 100:.1f}%）")
print("   帯域が狭い（低い帯域）ほど、1本の応答の中の「独立なゆらぎ」の数が少なく、ばらつきが大きい")

# ---------------------------------------------------------------- 図
h = R.polack_rir(T60, FS, SEC, rng=0, direct=False, pnr_db=45)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
t = np.arange(len(h)) / FS
p_db = 10 * np.log10(h ** 2 / np.max(h ** 2) + 1e-30)
ax[0].plot(t, p_db, color="0.75", lw=0.4, label="h² [dB]")
for m, col in zip(methods, ("C3", "C1", "C0")):
    r = R.rt60_from_ir(h, FS, "T20", noise=m)
    D = r["D"]
    ax[0].plot(np.arange(len(D)) / FS, D, color=col, label=f"{methods[m]}（T20 = {r['T']:.2f} s）")
info = R.lundeby(h, FS)
ax[0].axvline(info["t_cross"] / FS, color="C0", ls=":", label="クロスポイント")
ax[0].plot([0, T60], [0, -60], "k--", lw=1, label="真の減衰（0.5 s）")
ax[0].set_ylim(-80, 3)
ax[0].set_xlabel("時間 [s]")
ax[0].set_ylabel("dB")
ax[0].set_title("PNR 45 dB の応答の EDC")
ax[0].legend(fontsize=8)
ax[1].hist(T, bins=20, alpha=0.7, label="100 本の T30")
ax[1].errorbar([T[0]], [2], xerr=[1.96 * SE[0]], fmt="o", color="C3", label="1本目の OLS の 95% 区間")
ax[1].set_xlabel("T30 [s]")
ax[1].set_title("当てはめの標準誤差は、本当のばらつきよりずっと小さい")
ax[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "noise_floor.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print("\n図：out/noise_floor.png")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
l45 = re.search(r"PNR 45 dB：そのまま T20 [\d.±]+、T30 ([\d.]+)±.*打ち切り＋補正 T20 [\d.±]+、T30 ([\d.]+)±", cap.stdout)
ok(float(l45.group(1)) > 3 and abs(float(l45.group(2)) - 0.5) < 0.01, f"[1] PNR 45 dB：そのままの T30 {l45.group(1)} s（大きく外れる）、打ち切り＋補正 {l45.group(2)} s")
ok(re.search(r"PNR 35 dB：.*打ち切り＋補正 T20 [\d.±]+、T30\s+——", cap.stdout) is not None, "[1] PNR 35 dB で打ち切り＋補正の T30 が「——」（測れない）")
ok(num(r"本当のばらつきの 1/(\d+)") > 10, "[2] OLS の標準誤差が本当のばらつきよりずっと小さい（教科書 1/34）")
ok(num(r"含んだ割合：(\d+)%") < 30, "[2] OLS の 95% 区間が真の値を含んだ割合がずっと低い（教科書 4%）")
cv = [num(rf"{f} Hz：.*?変動係数 ([\d.]+)%") for f in (250, 1000, 4000)]
ok(cv[0] > cv[1] > cv[2], f"[3] 変動係数が 250 Hz {cv[0]}%・1 kHz {cv[1]}%・4 kHz {cv[2]}%（低い帯域ほど大きい）")

## 6. ESS・TSP・非同期の録音のシミュレーション（IV-3-3-11、手順 E、`ess_sim.py`）

図の左のスペクトログラムに、基本波の上に 2f・3f の薄い線が見え、中央で負の時刻に山が並ぶことを確かめてください。最後にスイープ（最大 0.1、5 秒）を聞きます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-3 E：指数スイープ（ESS）と TSP をシミュレーションで確かめる（IV-3-3-11）

  uv run python ess_sim.py

  [1] ESS の性質：瞬時周波数、1オクターブにかかる時間、スペクトルの傾き（−3 dB/oct）
  [2] ひずみのあるスピーカ（x + a₂x² + a₃x³ → 帯域通過）＋ 部屋 ＋ 雑音 を ESS で測る：
      高調波の応答が、予想した負の時刻 −Δt_k に現れるか。線形の応答は正しく取り出せるか
  [3] 同じ系を TSP（同期加算）で測る：ひずみはどこへ行くか
  [4] 録音側の標本化周波数のずれ（非同期の録音、IV-5-3-10）が ESS の測定に与える影響
wav：out/ess_sweep.wav（聞く用、最大 0.1）  図：out/ess_sim.png
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
import numpy as np
import pyroomacoustics as pra
from scipy.signal import butter, sosfilt
from common import plotting  # noqa: F401
from common import room as R
from common.array import sfo_resample
from common.audio_io import save
from common.filters import conv_fft
from common.stft import stft

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
FS = 16000
F1, F2, T = 50.0, 7000.0, 5.0
A2, A3 = 0.1, 0.05                      # スピーカのひずみ（2次・3次）
rng = np.random.default_rng(0)

# ---------------------------------------------------------------- [1]
x, v = R.ess(F1, F2, T, FS)
Rl = np.log(F2 / F1)
print(f"[1] ESS：{F1:.0f} Hz → {F2:.0f} Hz を {T} 秒、R = ln(f2/f1) = {Rl:.3f}")
for tt in (0.0, 1.0, 2.5, 4.0, 5.0):
    print(f"   t = {tt:3.1f} s の瞬時周波数 f1·e^(tR/T) = {F1 * np.exp(tt * Rl / T):7.1f} Hz")
print(f"   1オクターブにかかる時間 T·ln2/R = {T * np.log(2) / Rl:.3f} s（どのオクターブも同じ）")
Xs = np.abs(np.fft.rfft(x)) ** 2
fr = np.fft.rfftfreq(len(x), 1 / FS)
lev = {f: 10 * np.log10(np.mean(Xs[(fr > f / 1.2) & (fr < f * 1.2)])) for f in (125, 250, 500, 1000, 2000, 4000)}
print("   パワースペクトル（250 Hz を 0 dB）：" + "、".join(f"{f} Hz {lev[f] - lev[250]:+.1f}" for f in lev)
      + " → 1オクターブで約 −3 dB（ピンクノイズと同じ形）")
for k in (2, 3, 4, 5):
    print(f"   {k} 次の高調波の応答の位置 −Δt_{k} = −T ln{k}/R = {-R.harmonic_delay(k, F1, F2, T) * 1e3:8.1f} ms")
save(OUT / "ess_sweep.wav", x * 0.1, FS)

# ---------------------------------------------------------------- 測る系（スピーカ → 部屋）
sp_sos = butter(2, [150, 6000], btype="bandpass", fs=FS, output="sos")     # 小さなスピーカの帯域のつもり
dim = [4.0, 3.5, 2.5]
alpha, mo = pra.inverse_sabine(0.4, dim)
room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(alpha), max_order=mo, air_absorption=False)
room.add_source([1.1, 1.3, 1.0])
room.add_microphone([2.05, 1.62, 1.12])
room.compute_rir()
h_room = room.rir[0][0].astype(float)
imp = np.zeros(len(h_room) + 2000)
imp[0] = 1.0
h_true = conv_fft(sosfilt(sp_sos, imp), h_room)[:len(imp)]       # 線形の部分の本当の応答（スピーカ × 部屋）


def system(sig, a2=A2, a3=A3, noise_db=-70):
    """ハマースタイン型：静的な非線形 → スピーカの帯域 → 部屋 → 雑音"""
    y = sig + a2 * sig ** 2 + a3 * sig ** 3
    y = conv_fft(sosfilt(sp_sos, np.r_[y, np.zeros(len(h_room))]), h_room)[:len(sig) + len(h_room)]
    return y + 10 ** (noise_db / 20) * rng.standard_normal(len(y))


P = 4000
ref = np.r_[np.zeros(P), h_true, np.zeros(P)]


def band_err_db(est, lo=200, hi=5000):
    """lo〜hi [Hz] で、推定した応答と本当の応答（大きさをそろえた後）の誤差 [dB]。戻り値：(誤差, 大きさの比)"""
    E, Tt = np.fft.rfft(est, len(ref)), np.fft.rfft(ref)
    f = np.fft.rfftfreq(len(ref), 1 / FS)
    k = (f >= lo) & (f <= hi)
    g = np.real(np.vdot(Tt[k], E[k]) / np.vdot(Tt[k], Tt[k]))         # 最小二乗の大きさ
    return 10 * np.log10(np.sum(np.abs(E[k] - g * Tt[k]) ** 2) / np.sum(np.abs(g * Tt[k]) ** 2)), g


# ---------------------------------------------------------------- [2] ESS
print(f"\n[2] ESS で測る（スピーカのひずみ a₂ = {A2}、a₃ = {A3}、雑音 −70 dB、部屋 {dim}、RT60 0.4 s の設定）")
y = system(x)
n_pre = int(1.5 * FS)
h_all, i0 = R.ess_deconvolve(y, v, n_pre=n_pre, n_post=len(h_true) + P)
lin = h_all[i0 - P:i0 + len(h_true) + P]
pk_lin = np.max(np.abs(lin))
i_lin = i0 - P + int(np.argmax(np.abs(lin)))                  # 線形の応答の山（直接音）の位置
print(f"   線形の応答の山：時刻 0 から {(i_lin - i0) / FS * 1e3:.2f} ms（音が届くまでの時間と、pra の 40 標本の遅れ）")
for k in (2, 3):
    dt = R.harmonic_delay(k, F1, F2, T)
    j = i_lin - int(round(dt * FS))
    seg = np.abs(h_all[j - 300:j + 300])
    print(f"   {k} 次：線形の山から見て 予想 −{dt * 1e3:.1f} ms、見つけた山 −{(i_lin - (j - 300 + np.argmax(seg))) / FS * 1e3:.1f} ms、"
          f"山の高さ（線形の山を 0 dB）{20 * np.log10(seg.max() / pk_lin):+.1f} dB")
print(f"   予想：2次の応答の大きさ a₂/2 = {A2 / 2:.3f}、3次 a₃/4 = {A3 / 4:.4f}（線形の 1 + 3a₃/4 = {1 + 0.75 * A3:.4f} に対して"
      f" {20 * np.log10(A2 / 2 / (1 + 0.75 * A3)):+.1f}・{20 * np.log10(A3 / 4 / (1 + 0.75 * A3)):+.1f} dB）")
e_ess, g_ess = band_err_db(lin)
print(f"   線形の応答：大きさ {g_ess:.4f} 倍（予想 1 + 3a₃/4 = {1 + 0.75 * A3:.4f}）、200 Hz〜5 kHz の形の誤差 {e_ess:.1f} dB")
r_true = R.rt60_from_ir(h_true, FS, "T20", noise="none")["T"]
r_ess = R.rt60_from_ir(lin[P:], FS, "T20", noise="lundeby")["T"]
print(f"   T20：本当の応答 {r_true:.3f} s、ESS で測った応答 {r_ess:.3f} s")

# ---------------------------------------------------------------- [3] TSP
print("\n[3] TSP（2^16 点 = 4.1 秒の周期を 3 回、最初の1回を捨てて2回を同期加算）")
Nt = 1 << 16
xt, Xt = R.tsp(Nt)
for a2, a3, nm in ((0.0, 0.0, "ひずみなし"), (A2, A3, "ひずみあり")):
    yt = system(np.tile(xt, 3), a2, a3)[:3 * Nt]
    ht = R.tsp_deconvolve(yt, Xt, Nt, 3)
    est = np.r_[ht[-P:], ht[:len(h_true) + P]]                  # 円状なので、負の時刻は周期の終わりにある
    e_t, g_t = band_err_db(est)
    tail = ht[len(h_true) + 2000:Nt - P]                         # 応答が終わったあとの区間（本来は雑音だけ）
    print(f"   {nm}：形の誤差 {e_t:.1f} dB、応答のあとの区間のレベル（線形の山を 0 dB）"
          f"{10 * np.log10(np.mean(tail ** 2) / np.max(ht ** 2)):.1f} dB")
print("   TSP ではひずみの成分が周期全体に広がり、線形の応答に重なる（ESS のように分けられない）")

# ---------------------------------------------------------------- [4] 非同期の録音
print("\n[4] 録音機の標本化周波数が fs(1+ε) のとき（スマホで録った場合。開始のずれは山の位置で吸収される）")
y0 = system(x, 0.0, 0.0, noise_db=-90)
def band_delay_us(hh, hb, fc):
    """hh が hb より何 µs 遅れているか（fc ± 10% の帯域の、相互スペクトルの位相の傾きから）"""
    A, B = np.fft.rfft(hh), np.fft.rfft(hb)
    f = np.fft.rfftfreq(len(hb), 1 / FS)
    k = (f > fc / 1.1) & (f < fc * 1.1)
    slope = np.polyfit(f[k], np.unwrap(np.angle(A[k] * np.conj(B[k]))), 1)[0]
    return -slope / (2 * np.pi) * 1e6


h_base, _ = R.ess_deconvolve(y0, v, n_pre=P, n_post=len(h_true) + P)
for eps in (20e-6, 50e-6, 100e-6):
    yy, eps_used = sfo_resample(y0, eps)
    hh, j0 = R.ess_deconvolve(yy, v, n_pre=P, n_post=len(h_true) + P)
    d = {fc: band_delay_us(hh, h_base, fc) for fc in (250, 1000, 4000)}
    pred = eps * T * np.log(4000 / 250) / Rl * 1e6
    t20 = R.rt60_from_ir(hh[int(np.argmax(np.abs(hh))):], FS, "T20", noise="lundeby")["T"]
    print(f"   ε = {eps * 1e6:5.1f} ppm：ずれなしに対する遅れ 250 Hz {d[250]:6.1f}・1 kHz {d[1000]:6.1f}・4 kHz {d[4000]:6.1f} µs。"
          f"4 kHz − 250 Hz = {d[4000] - d[250]:6.1f} µs（予想 ε·T·ln(4000/250)/R = {pred:6.1f}）、T20 {t20:.3f} s")
print("   周波数 f の成分の到着は +ε·t_f（t_f = T ln(f/f1)/R：スイープが f を通る時刻）だけ遅れる。高い周波数ほど遅れる")

# ---------------------------------------------------------------- 図
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
Xstft = stft(y, L=1024, S=256)
ax[0].imshow(20 * np.log10(np.abs(Xstft) + 1e-6), origin="lower", aspect="auto", cmap="magma",
             extent=(0, Xstft.shape[1] * 256 / FS, 0, FS / 2), vmin=20 * np.log10(np.abs(Xstft).max()) - 80)
tt = np.linspace(0, T, 200)
for k in (1, 2, 3):
    ax[0].plot(tt, k * F1 * np.exp(tt * Rl / T), "--", lw=0.8, color="w" if k == 1 else "c")
ax[0].set_ylim(0, FS / 2)
ax[0].set_xlabel("時間 [s]")
ax[0].set_ylabel("周波数 [Hz]")
ax[0].set_title("ひずみのある系の録音（破線：k f(t)）")
th = (np.arange(len(h_all)) - i0) / FS
ax[1].plot(th, 20 * np.log10(np.abs(h_all) / pk_lin + 1e-9), lw=0.5)
for k in (2, 3, 4):
    ax[1].axvline(-R.harmonic_delay(k, F1, F2, T), color="C3", ls=":", lw=1)
ax[1].set_ylim(-100, 3)
ax[1].set_xlabel("時間 [s]（0 が線形の応答）")
ax[1].set_ylabel("dB（線形の山を 0）")
ax[1].set_title("ESS の逆フィルタの出力：高調波は負の時刻に分かれる")
tt2 = np.arange(len(h_true)) / FS
ax[2].plot(tt2 * 1e3, h_true / np.max(np.abs(h_true)), label="本当の応答", lw=1)
ax[2].plot(tt2 * 1e3, lin[P:P + len(h_true)] / g_ess / np.max(np.abs(h_true)), "--", label="ESS で測った応答", lw=1)
ax[2].set_xlim(0, 60)
ax[2].set_xlabel("時間 [ms]")
ax[2].set_title("線形の応答の始まり（大きさをそろえて重ねた）")
ax[2].legend()
fig.tight_layout()
fig.savefig(OUT / "ess_sim.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print("\nwav：out/ess_sweep.wav（最大 0.1）　図：out/ess_sim.png")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
ok(abs(num(r"1オクターブにかかる時間 T·ln2/R = ([\d.]+) s") - 0.701) < 0.001, "[1] 1オクターブ 0.701 s")
ok(abs(num(r"500 Hz ([-+\d.]+)、1000") + 3.0) < 0.2, "[1] スペクトルが1オクターブで −3.0 dB")
ok(abs(num(r"2 次：.*?見つけた山 ([-−\d.]+) ms") + 701.4) < 1, "[2] 2次の山が −701.4 ms")
ok(abs(num(r"3 次：.*?見つけた山 ([-−\d.]+) ms") + 1111.6) < 1, "[2] 3次の山が −1111.6 ms")
ok(abs(num(r"大きさ ([\d.]+) 倍") - 1.0374) < 0.002, "[2] 線形の応答の大きさ 1.0374 倍")
ok(num(r"ひずみあり：.*?区間のレベル（線形の山を 0 dB）([-\d.]+) dB") < -50, "[3] TSP のひずみありで応答のあとの区間が −66.5 dB 前後")
ok(abs(num(r"ε =  50\.0 ppm：.*?4 kHz − 250 Hz =\s+([\d.]+) µs") - 135) < 10, "[4] 50 ppm で 4 kHz − 250 Hz の遅れ 135 µs 前後（予想 140）")

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）。低い音から高い音へ「同じ速さで」上がっていくように聞こえる
for f in ['ess_sweep.wav']:
    print(f)
    listen_file("out/" + f)

## 7. 合成の録音でインパルス応答を測る（IV-3-3-11、手順 F の前の試し、`measure_ir.py --simulate`）

実際に Mac で鳴らして録る `--mac`、スマホ用のスイープを作る `--write-sweep`、スマホの録音を解析する `--analyze` は、このノートブックには入れていません（Mac のターミナルで実行します）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['measure_ir.py', '--simulate', '--label', 'sim']   # ノートブック版：ターミナルでの引数の代わり
"""IV-3 F・G：自分の部屋のインパルス応答を指数スイープ（ESS）で測り、残響時間を求める

  uv run python measure_ir.py --simulate --label sim               機材なしで全体を試す（合成の録音）
  uv run python measure_ir.py --mac --label desk --level -30       Mac のスピーカで鳴らし、内蔵マイクで同時に録る
  uv run python measure_ir.py --write-sweep                        スマホで録るためのスイープを out/sweep_48k.wav に書く
  uv run python measure_ir.py --analyze data/phone_P1.wav --label P1   スマホで録った wav を解析する

  スイープ：100 Hz → 16 kHz、5 秒、48 kHz。--repeat 回（既定 3）、間に 3 秒の無音をはさむ。
  音の大きさ：--level はスイープの最大値 [dBFS]。既定 −30 dBFS（最大 0.032）。−6 dBFS より大きくはできない。
             はじめは Mac の音量を小さくし、耳で「うるさくない」ことを確かめてから少しずつ上げる。
  結果：out/ir_results.csv（1行 = 1回の測定 × 1つの帯域）、out/ir_<label>_<回>.npy（インパルス応答）、
        out/ir_<label>.png（1回目の応答と帯域ごとの EDC）
"""
import argparse
import pathlib
import sys
import time

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import plotting  # noqa: F401
from common import room as R
from common.audio_io import load, save

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
FS, F1, F2, T, GAP, PRE = 48000, 100.0, 16000.0, 5.0, 3.0, 1.0
BANDS = (250, 500, 1000, 2000, 4000)
IR_SEC = 1.5                                            # 取り出すインパルス応答の長さ [s]（RT60 の 2〜3 倍）

ap = argparse.ArgumentParser()
g = ap.add_mutually_exclusive_group(required=True)
g.add_argument("--simulate", action="store_true", help="合成の録音で試す")
g.add_argument("--mac", action="store_true", help="Mac のスピーカと内蔵マイクで測る")
g.add_argument("--write-sweep", action="store_true", help="スマホで録るためのスイープの wav を書く")
g.add_argument("--analyze", metavar="WAV", help="録った wav を解析する")
ap.add_argument("--label", default="test", help="測定の名前（場所など。例 P1）")
ap.add_argument("--level", type=float, default=-30.0, help="スイープの最大値 [dBFS]（既定 −30）")
ap.add_argument("--repeat", type=int, default=3, help="スイープのくり返しの回数")
a = ap.parse_args()
if a.level > -6:
    sys.exit("--level は −6 dBFS 以下にしてください（スピーカと耳を守るため。ふつうは −30〜−20 で足りる）")

x, v = R.ess(F1, F2, T, FS)
amp = 10 ** (a.level / 20)
period = int((T + GAP) * FS)


def build_signal():
    """先頭 PRE 秒の無音 → （スイープ → GAP 秒の無音）× repeat"""
    sig = np.zeros(int(PRE * FS) + a.repeat * period)
    for j in range(a.repeat):
        s0 = int(PRE * FS) + j * period
        sig[s0:s0 + len(x)] = amp * x
    return sig


def simulate_recording(sig):
    """合成の録音：6畳ほどの部屋（3.6×2.7×2.4 m、RT60 0.4 s の設定）、ひずみのあるスピーカ、
    録音の最大 −20 dBFS、暗騒音 −60 dBFS、録音機の標本化周波数のずれ 30 ppm、開始のずれ 0.37 秒"""
    import pyroomacoustics as pra
    from scipy.signal import butter, sosfilt
    from common.array import sfo_resample
    from common.filters import conv_fft
    rng = np.random.default_rng(0)
    dim = [3.6, 2.7, 2.4]
    alpha, mo = pra.inverse_sabine(0.4, dim)
    room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(alpha), max_order=mo, air_absorption=True)
    room.add_source([0.6, 0.9, 0.8])
    room.add_microphone([1.9, 1.7, 1.1])
    room.compute_rir()
    h = room.rir[0][0].astype(float)
    sp = butter(2, [180, 15000], btype="bandpass", fs=FS, output="sos")    # 小さなスピーカのつもり
    s = sig / amp * 0.5
    s = (s + 0.05 * s ** 2 + 0.02 * s ** 3) * amp / 0.5
    y = conv_fft(sosfilt(sp, s), h)[:len(sig)]
    y = y / np.max(np.abs(y)) * 10 ** (-20 / 20)              # 録音の最大を −20 dBFS に
    y = y + 10 ** (-60 / 20) * rng.standard_normal(len(y))   # 暗騒音とマイクの雑音のつもり
    y, _ = sfo_resample(y, 30e-6)
    return np.r_[np.zeros(int(0.37 * FS)), y]


# ---------------------------------------------------------------- 録る・書く
if a.write_sweep:
    sig = build_signal()
    p = save(OUT / "sweep_48k.wav", sig, FS)
    print(f"{p} を書きました（{len(sig) / FS:.1f} 秒、最大 {a.level:.0f} dBFS、スイープ {a.repeat} 回）。")
    print("スマホの録音を始めてから、afplay で鳴らす：  afplay out/sweep_48k.wav")
    print("録った音を wav にして（afconvert -f WAVE -d LEI16 入力.m4a data/phone_P1.wav）、--analyze で解析する")
    sys.exit(0)

if a.simulate:
    rec = simulate_recording(build_signal())
    src_name = "sim"
elif a.mac:
    import sounddevice as sd
    sig = build_signal()
    print(f"出力：{sd.query_devices(kind='output')['name']}、入力：{sd.query_devices(kind='input')['name']}")
    print(f"{len(sig) / FS:.0f} 秒鳴らします（最大 {a.level:.0f} dBFS）。うるさいと感じたら Ctrl+C で止め、音量を下げる")
    time.sleep(1.0)
    rec = sd.playrec((sig[:, None]).astype(np.float32), samplerate=FS, channels=1, dtype="float32")
    sd.wait()
    rec = rec[:, 0].astype(float)
    save(OUT / f"rec_{a.label}.wav", np.clip(rec, -1, 1), FS)
    src_name = "mac"
else:
    rec, fs0 = load(a.analyze, fs=FS)
    if rec.ndim == 2:
        rec = rec[:, 0]
    src_name = "file"
print(f"録音：{len(rec) / FS:.1f} 秒、最大 {20 * np.log10(np.max(np.abs(rec)) + 1e-12):.1f} dBFS")
if np.max(np.abs(rec)) > 0.99:
    print("注意：録音が割れています（最大 0 dBFS）。--level を下げるか、録音の音量を下げて測り直す")

# ---------------------------------------------------------------- 逆フィルタで応答を取り出す
from common.filters import conv_fft
g_all = conv_fft(rec, v)                                 # 線形の応答は、スイープの始まり + len(v) − 1 のあたり
z = len(v) - 1
# 1回目の山：全体の最大（高調波の山は線形の山より小さいので、最大は線形の山のどれか）。そこから前後にくり返しを探す
i_max = int(np.argmax(np.abs(g_all)))
peaks = []
for j in range(-a.repeat, a.repeat + 1):
    c = i_max + j * period
    if c - int(0.05 * FS) < z or c + int(IR_SEC * FS) > len(g_all):
        continue
    seg = np.abs(g_all[c - int(0.05 * FS):c + int(0.05 * FS)])
    k = c - int(0.05 * FS) + int(np.argmax(seg))
    if seg.max() > 0.3 * np.abs(g_all[i_max]):
        peaks.append(k)
peaks = sorted(set(peaks))
print(f"見つけたスイープ：{len(peaks)} 回（山の間隔 {np.diff(peaks).mean() / FS if len(peaks) > 1 else float('nan'):.5f} s、"
      f"名目 {T + GAP:.1f} s）")
if len(peaks) > 1:
    eps_hat = np.diff(peaks).mean() / period - 1
    print(f"   間隔から見た標本化周波数のずれの目安 {eps_hat * 1e6:+.0f} ppm（1 標本 = {1 / period * 1e6:.1f} ppm の刻み。"
          "Mac の同時録音ならほぼ 0）")

rows = []
pre = int(0.005 * FS)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for rep, k in enumerate(peaks):
    h = g_all[k - pre:k + int(IR_SEC * FS)]
    np.save(OUT / f"ir_{a.label}_{rep}.npy", h)
    hd2 = g_all[k - int(R.harmonic_delay(2, F1, F2, T) * FS) - 200:k - int(R.harmonic_delay(2, F1, F2, T) * FS) + 200]
    info = R.lundeby(h[pre:], FS)
    pnr = 10 * np.log10(np.max(h ** 2) / info["noise"])
    line = f"[{rep}] 山 {k / FS:.3f} s、PNR {pnr:.1f} dB、2次ひずみの山 {20 * np.log10(np.max(np.abs(hd2)) / np.max(np.abs(h))):+.1f} dB"
    for band in ("全帯域",) + BANDS:
        hb = h if band == "全帯域" else R.octave_band(h, FS, band)
        res = {m: R.rt60_from_ir(hb, FS, m, noise="lundeby")["T"] for m in ("EDT", "T20", "T30")}
        rows.append(dict(label=a.label, source=src_name, rep=rep, band=str(band), EDT=res["EDT"], T20=res["T20"],
                         T30=res["T30"], pnr_db=pnr, drr_db=R.drr_db(h, FS), c50_db=R.clarity_db(h, FS)))
        if band in (500, 1000, 2000):
            line += f"、{band} Hz の T20 {res['T20']:.3f} s"
        if rep == 0 and band != "全帯域":
            r = R.rt60_from_ir(hb, FS, "T20", noise="lundeby")
            ax[1].plot(np.arange(len(r["D"])) / FS, r["D"], label=f"{band} Hz（T20 {r['T']:.2f} s）")
    print(line)
    if rep == 0:
        tt = (np.arange(len(h)) - pre) / FS * 1e3
        ax[0].plot(tt, 20 * np.log10(np.abs(h) / np.max(np.abs(h)) + 1e-9), lw=0.4)
df = pd.DataFrame(rows)
csv = OUT / "ir_results.csv"
df.to_csv(csv, mode="a", header=not csv.exists(), index=False)
print(f"\n{csv} に {len(df)} 行を足しました（同じ label で測り直すと行が増える。くり返しは rep）")
summ = df[df.band != "全帯域"].groupby("band", sort=False)[["T20", "T30"]].agg(["mean", "std"])
print("この測定の帯域ごとの T20・T30（くり返しの平均と標準偏差。同じ場所のくり返しなので、部屋の T60 のばらつきではない）")
print(summ.round(3).to_string())
ax[0].set_xlim(-5, 400)
ax[0].set_ylim(-90, 3)
ax[0].set_xlabel("時間 [ms]")
ax[0].set_ylabel("|h| [dB]（山を 0）")
ax[0].set_title(f"{a.label}：インパルス応答（1回目）")
ax[1].set_ylim(-60, 2)
ax[1].set_xlabel("時間 [s]")
ax[1].set_title("帯域ごとの EDC（Lundeby の打ち切り＋補正）")
ax[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / f"ir_{a.label}.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print(f"図：out/ir_{a.label}.png")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
ok("見つけたスイープ：3 回" in cap.stdout, "見つけたスイープ：3 回")
ok(abs(num(r"ずれの目安 ([-+\d]+) ppm") - 30) <= 3, "間隔から見た標本化周波数のずれ +30 ppm（合成の録音に入れた値）")
ok(abs(num(r"\[0\] 山 .*?PNR ([\d.]+) dB") - 63) < 3, "PNR 約 63 dB")
ok(abs(num(r"\[0\] 山 .*?2次ひずみの山 ([-\d.]+) dB") + 36) < 3, "2次ひずみの山 約 −36 dB")
ok(abs(num(r"\[0\] 山 .*?1000 Hz の T20 ([\d.]+) s") - 0.42) < 0.02, "1 kHz の T20 0.42 s 前後")

## 8. 場所を単位にした T60 の集計（IV-3-3-12、手順 G の集計の前の試し、`t60_stats.py --simulate`）

合成の測定（6 か所 × 3 回）で、疑似反復の誤り（[1]）と正しい集計（[2]）を比べます。自分の測定の CSV（`--csv out/ir_results.csv`）は Mac で集計してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['t60_stats.py', '--simulate']   # ノートブック版：ターミナルでの引数の代わり
"""IV-3 H：部屋の T60 を、場所のばらつきとくり返しを分けて報告する（IV-3-3-12）

  uv run python t60_stats.py --simulate                    合成の測定（6 か所 × 3 回）で試す
  uv run python t60_stats.py --csv out/ir_results.csv      measure_ir.py の結果を集計する（label = 場所）

  [1] 誤り：すべての行（場所 × くり返し）を独立なデータとみなして平均と 95% 区間を出す（疑似反復）
  [2] 正しい：場所ごとに平均してから、場所の数で区間を出す
  [3] 線形混合モデル（場所を変量効果）で、場所の間のばらつきと、くり返しの間のばらつきを分ける（ICC）
  [4] 誤り：1 か所の測定の値を「部屋の T60」として報告する
"""
import argparse
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from common import room as R

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
ap = argparse.ArgumentParser()
g = ap.add_mutually_exclusive_group(required=True)
g.add_argument("--simulate", action="store_true")
g.add_argument("--csv")
ap.add_argument("--band", default="1000", help="集計する帯域（250・500・1000・2000・4000・全帯域）")
ap.add_argument("--metric", default="T20", choices=["EDT", "T20", "T30"])
a = ap.parse_args()


def simulate():
    """3.6×2.7×2.4 m（RT60 0.4 s の設定）で、音源は1か所、マイクを 6 か所に置き、各 3 回測ったつもりのデータ。
    くり返しの違いは測定の雑音（PNR 55 dB）だけ。帯域は 1 kHz"""
    import pyroomacoustics as pra
    fs = 16000
    dim = [3.6, 2.7, 2.4]
    alpha, mo = pra.inverse_sabine(0.4, dim)
    rng = np.random.default_rng(3)
    mics = [[1.9, 1.7, 1.1], [2.9, 0.6, 1.2], [3.1, 2.2, 0.9], [1.2, 2.3, 1.3], [2.4, 1.1, 1.6], [0.8, 0.5, 1.0]]
    rows = []
    for i, m in enumerate(mics):
        room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(alpha), max_order=mo, air_absorption=True)
        room.add_source([0.6, 0.9, 0.8])
        room.add_microphone(m)
        room.compute_rir()
        h = room.rir[0][0].astype(float)
        h = h[np.argmax(np.abs(h)):]
        hb0 = R.octave_band(h, fs, 1000)
        for rep in range(3):
            noise = rng.standard_normal(len(hb0)) * np.max(np.abs(hb0)) * 10 ** (-55 / 20)
            r = R.rt60_from_ir(hb0 + R.octave_band(noise, fs, 1000), fs, a.metric, noise="lundeby")
            rows.append(dict(label=f"P{i + 1}", rep=rep, band="1000", **{a.metric: r["T"]}))
    return pd.DataFrame(rows)


df = simulate() if a.simulate else pd.read_csv(a.csv)
df = df[df.band.astype(str) == a.band].dropna(subset=[a.metric])
y = a.metric
n_pos, n_all = df.label.nunique(), len(df)
print(f"帯域 {a.band} Hz の {y}：場所 {n_pos} か所、全部で {n_all} 行")
print(df.groupby("label")[y].agg(["mean", "std", "count"]).round(4).to_string())
if n_pos < 2:
    sys.exit("場所が1か所だけなので、部屋の T60 の区間は出せません（[4] の誤りになる）。場所を変えて測る")

# ---------------------------------------------------------------- [1] 誤り
m, s = df[y].mean(), df[y].std(ddof=1)
h1 = stats.t.ppf(0.975, n_all - 1) * s / np.sqrt(n_all)
print(f"\n[1] 誤り（全部の行を独立とみなす）：平均 {m:.3f} s、95% 区間 {m - h1:.3f}〜{m + h1:.3f}（n = {n_all}）")

# ---------------------------------------------------------------- [2] 正しい
pm = df.groupby("label")[y].mean()
h2 = stats.t.ppf(0.975, n_pos - 1) * pm.std(ddof=1) / np.sqrt(n_pos)
print(f"[2] 正しい（場所ごとに平均してから）：平均 {pm.mean():.3f} s、95% 区間 {pm.mean() - h2:.3f}〜{pm.mean() + h2:.3f}"
      f"（n = {n_pos} か所）。区間の幅は [1] の {h2 / h1:.1f} 倍")

# ---------------------------------------------------------------- [3] 混合モデル
md = smf.mixedlm(f"{y} ~ 1", df, groups=df["label"]).fit(reml=True)
var_pos = float(md.cov_re.iloc[0, 0])
var_rep = float(md.scale)
icc = var_pos / (var_pos + var_rep)
k = n_all / n_pos
print(f"[3] 線形混合モデル：場所の間の標準偏差 {np.sqrt(var_pos):.4f} s、くり返しの間の標準偏差 {np.sqrt(var_rep):.4f} s、"
      f"ICC = {icc:.2f}")
print(f"    有効なデータの数 n/(1+(k−1)ICC) = {n_all / (1 + (k - 1) * icc):.1f}（見かけの {n_all}、場所の数 {n_pos}）")
print("    くり返しを増やしても減るのは「くり返しの間」のばらつきだけ。部屋の T60 を知るには場所を増やす")

# ---------------------------------------------------------------- [4] 1か所
print(f"[4] 誤り：1 か所だけで測ると、場所の平均は {pm.min():.3f}〜{pm.max():.3f} s のどれにもなりえた"
      f"（全体の平均との差は最大 {np.max(np.abs(pm - pm.mean())) / pm.mean() * 100:.1f}%）")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
w1 = re.search(r"\[1\].*?95% 区間 ([\d.]+)〜([\d.]+)", cap.stdout)
w2 = re.search(r"\[2\].*?95% 区間 ([\d.]+)〜([\d.]+)", cap.stdout)
ok(abs(float(w1.group(1)) - 0.398) < 0.005 and abs(float(w1.group(2)) - 0.426) < 0.005, f"[1] 誤った区間 {w1.group(1)}〜{w1.group(2)} s（教科書 0.398〜0.426）")
ok(abs(float(w2.group(1)) - 0.380) < 0.005 and abs(float(w2.group(2)) - 0.444) < 0.005, f"[2] 正しい区間 {w2.group(1)}〜{w2.group(2)} s（教科書 0.380〜0.444）")
ok(num(r"ICC = ([\d.]+)") > 0.95 and abs(num(r"= ([\d.]+)（見かけの 18") - 6.0) < 0.2, "[3] ICC ≈ 1.00、有効なデータの数 6.0")

## 9. 部屋のモードと指向性（IV-3-3-9・3-10、手順 H、`modes_directivity.py`）

引数を省くと 6 畳（3.6×2.7×2.4 m）・T60 0.4 s の例になります。自分の部屋では `sys.argv` に `"--dim", "縦", "横", "高さ", "--rt60", "測った値"` を足してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['modes_directivity.py']   # ノートブック版：ターミナルでの引数の代わり
"""IV-3 I：部屋のモードとシュレーダー周波数、マイクの指向性と臨界距離（IV-3-3-9・3-10）

  uv run python modes_directivity.py                         6 畳ほどの部屋（3.6×2.7×2.4 m、T60 0.4 s）
  uv run python modes_directivity.py --dim 4.5 3.6 2.4 --rt60 0.5   自分の部屋の大きさと測った T60

  [1] 低い固有周波数の一覧（軸・接線・斜めのモード）
  [2] モードの数を数え、式 N(f) ≈ 4πVf³/(3c³) + πSf²/(4c²) + Lf/(8c) と比べる。シュレーダー周波数
  [3] 1次の指向性（無指向・単一指向・双指向など）の指向係数 Q と、臨界距離への効き方
図：out/modes_directivity.png
"""
import argparse
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
import numpy as np
from common import plotting  # noqa: F401
from common import room as R

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
ap = argparse.ArgumentParser()
ap.add_argument("--dim", type=float, nargs=3, default=[3.6, 2.7, 2.4])
ap.add_argument("--rt60", type=float, default=0.4)
a = ap.parse_args()
C = R.C_SOUND
V, S = R.room_volume_surface(a.dim)
Ltot = 4 * sum(a.dim)

# ---------------------------------------------------------------- [1]
modes = R.room_modes(a.dim, 200.0)
kind = {1: "軸", 2: "接線", 3: "斜め"}
print(f"[1] 部屋 {a.dim} m（V = {V:.1f} m³、S = {S:.1f} m²）の固有周波数（低いほうから 12 個）")
for f, nx, ny, nz in modes[:12]:
    print(f"   {f:6.1f} Hz　(nx, ny, nz) = ({int(nx)}, {int(ny)}, {int(nz)})　{kind[int(nx > 0) + int(ny > 0) + int(nz > 0)]}モード")

# ---------------------------------------------------------------- [2]
fs_ = R.schroeder_frequency(a.rt60, V)
print(f"\n[2] モードの数（剛い壁）：数えた値と式")
for f in (50, 100, 200, 300, 500):
    n_count = len(R.room_modes(a.dim, f))
    n_th = 4 * np.pi * V * f ** 3 / (3 * C ** 3) + np.pi * S * f ** 2 / (4 * C ** 2) + Ltot * f / (8 * C)
    print(f"   {f:4d} Hz まで：数えた {n_count:5d}　式 {n_th:7.1f}　（第1項だけ {4 * np.pi * V * f ** 3 / (3 * C ** 3):7.1f}）")
dens = 4 * np.pi * V * fs_ ** 2 / C ** 3
print(f"   シュレーダー周波数 2000√(T60/V) = {fs_:.0f} Hz（T60 {a.rt60} s）。そこでのモードの密度 {dens:.2f} 個/Hz、"
      f"モードの帯域幅 2.2/T60 = {2.2 / a.rt60:.1f} Hz → 1つの帯域幅に {dens * 2.2 / a.rt60:.1f} 個（係数を丸めたので 3 個より少し少ない）")
fs_exact = np.sqrt(3 * C ** 3 * a.rt60 / (4 * np.pi * 2.2 * V))
print(f"   （この節の導出どおり √(3c³T60/(4π·2.2V)) = {fs_exact:.0f} Hz なら、1つの帯域幅にちょうど 3 個。係数 2000 はこの 2092 を丸めた慣例の値）")
print("   この周波数より下ではモードが1つずつ目立ち、場所によって音の大きさが大きく変わる（統計的な扱いが成り立たない）")

# ---------------------------------------------------------------- [3]
print(f"\n[3] 指向性と臨界距離（T60 {a.rt60} s）")
rc1 = R.critical_distance(a.dim, a.rt60, 1.0)
for p, nm in ((1.0, "無指向"), (0.75, "サブカーディオイド"), (0.5, "単一指向（カーディオイド）"),
              (0.37, "スーパーカーディオイド（約）"), (0.25, "ハイパーカーディオイド"), (0.0, "双指向")):
    Q = R.directivity_factor(p)
    print(f"   {nm:16s} p = {p:4.2f}：Q = {Q:.2f}（DI {10 * np.log10(Q):4.2f} dB）、"
          f"後ろ（180°）の感度 {R.first_order_pattern(np.pi, p):+.2f}、臨界距離 {R.critical_distance(a.dim, a.rt60, Q):.2f} m"
          f"（無指向の {np.sqrt(Q):.2f} 倍）")
print(f"   無指向の臨界距離 {rc1:.2f} m：Mac から 1 m 離れて話すと、残響のほうが直接音より大きい")

# ---------------------------------------------------------------- 図
fig = plt.figure(figsize=(12, 4))
ax0 = fig.add_subplot(1, 2, 1, projection="polar")
th = np.linspace(0, 2 * np.pi, 361)
for p, nm in ((1.0, "無指向"), (0.5, "単一指向"), (0.25, "ハイパーカーディオイド"), (0.0, "双指向")):
    ax0.plot(th, np.abs(R.first_order_pattern(th, p)), label=nm)
ax0.set_title("1次の指向性 |p + (1−p)cosθ|")
ax0.legend(loc="lower left", fontsize=8, bbox_to_anchor=(-0.25, -0.15))
ax1 = fig.add_subplot(1, 2, 2)
ff = np.linspace(20, 400, 400)
ax1.plot(ff, [len(R.room_modes(a.dim, f)) for f in ff], label="数えたモードの数")
ax1.plot(ff, 4 * np.pi * V * ff ** 3 / (3 * C ** 3) + np.pi * S * ff ** 2 / (4 * C ** 2) + Ltot * ff / (8 * C), "--",
         label="式")
ax1.axvline(fs_, color="C3", ls=":", label=f"シュレーダー周波数 {fs_:.0f} Hz")
ax1.set_xlabel("周波数 [Hz]")
ax1.set_ylabel("その周波数までのモードの数")
ax1.legend()
fig.tight_layout()
fig.savefig(OUT / "modes_directivity.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print("\n図：out/modes_directivity.png")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
ok(abs(float(re.search(r"([\d.]+) Hz　\(nx, ny, nz\) = \(1, 0, 0\)", cap.stdout).group(1)) - 47.6) < 0.1, "[1] 最初のモード 47.6 Hz（軸モード）")
ok(num(r"100 Hz まで：数えた\s+(\d+)") == 7, "[2] 100 Hz までのモードの数 7 個（式 7.0）")
ok(num(r"2000√\(T60/V\) = (\d+) Hz") == 262, "[2] シュレーダー周波数 262 Hz")
ok(abs(num(r"単一指向（カーディオイド）.*?臨界距離 ([\d.]+) m") - 0.75) < 0.01 and abs(num(r"単一指向（カーディオイド）.*?無指向の ([\d.]+) 倍") - 1.73) < 0.01, "[3] カーディオイドの臨界距離 0.75 m（無指向の 1.73 倍）")

## 10. 確認問題の答え合わせ（手順 I、`quiz_check.py`）

確認問題（教科書の 7 節）は、**先に紙で解いてから**実行してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-3：確認問題の答えをコードで確かめる（IV-3-7）

  uv run python quiz_check.py
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と iv2/sounds.py はこのノートブックの隣
import numpy as np
from common import room as R

print("問1 音速")
for th in (15.0, 28.0):
    print(f"   {th} ℃：c = {float(R.speed_of_sound(th)):.2f} m/s")
c15, c28 = float(R.speed_of_sound(15.0)), float(R.speed_of_sound(28.0))
print(f"   5 m を進む時間の差 {(5 / c15 - 5 / c28) * 1e6:.1f} µs（16 kHz で {(5 / c15 - 5 / c28) * 16000:.2f} 標本）")

print("問2 音圧レベルの足し算")
print(f"   0.2 Pa（実効値）= {float(R.spl_db(0.2)):.2f} dB SPL")
print(f"   相関のない同じ大きさの2つ：+{10 * np.log10(2):.2f} dB、同じ位相の2つ：+{20 * np.log10(2):.2f} dB、"
      f"60 dB と 54 dB（相関なし）の和：{10 * np.log10(10 ** 6 + 10 ** 5.4):.2f} dB")

print("問3 距離減衰と臨界距離")
for r in (3.0, 8.0):
    print(f"   1 m で 70 dB の点音源、自由空間で {r} m：{70 - 20 * np.log10(r):.1f} dB")
rc = 0.43
for r in (0.2, 0.43, 1.0, 3.0):
    print(f"   r_c = 0.43 m の部屋で、直接音＋残響の全体（1 m の直接音を 0 dB）：r = {r} m で "
          f"{10 * np.log10(1 / r ** 2 + 1 / rc ** 2):+.1f} dB")

print("問4 サビンとアイリング（4.5×3.6×2.4 m、ᾱ = 0.15）")
dim = [4.5, 3.6, 2.4]
V, S = R.room_volume_surface(dim)
print(f"   V = {V:.2f} m³、S = {S:.2f} m²、A = {S * 0.15:.2f} m²")
print(f"   サビン {R.rt60_sabine(dim, 0.15):.3f} s、アイリング {R.rt60_eyring(dim, 0.15):.3f} s")
A_need = 24 * np.log(10) * V / (343 * 0.3)
print(f"   T = 0.3 s にするのに要る A = {A_need:.2f} m²（足りない分 {A_need - S * 0.15:.2f} m²。α = 0.8 の吸音材なら "
      f"{(A_need - S * 0.15) / (0.8 - 0.15):.1f} m²（もとの面を置きかえるので α の差 0.65 で割る））")

print("問5 サビンの式の係数")
print(f"   24 ln10 / c = {24 * np.log(10) / 343:.4f}（c = 343 m/s）、{24 * np.log(10) / float(R.speed_of_sound(20)):.4f}（c = 343.21）")

print("問6 シュレーダー法")
t5, t25 = 0.04, 0.21
print(f"   −5 dB が {t5} s、−25 dB が {t25} s：T20 = 3 × ({t25} − {t5}) = {3 * (t25 - t5):.3f} s")

print("問7 ESS（100 Hz → 16 kHz、5 秒）")
f1, f2, T = 100.0, 16000.0, 5.0
Rl = np.log(f2 / f1)
print(f"   1 kHz を通る時刻 T ln(1000/f1)/R = {T * np.log(1000 / f1) / Rl:.3f} s、1オクターブ {T * np.log(2) / Rl:.3f} s")
print(f"   Δt2 = {R.harmonic_delay(2, f1, f2, T):.3f} s、Δt3 = {R.harmonic_delay(3, f1, f2, T):.3f} s、"
      f"Δt3 − Δt2 = {R.harmonic_delay(3, f1, f2, T) - R.harmonic_delay(2, f1, f2, T):.3f} s")
print(f"   T60 = 0.5 s の部屋の応答を 60 dB まで2次と重ねないための最小の T：T ≥ 0.5·R/ln2 = {0.5 * Rl / np.log(2):.2f} s")

print("問8 鏡像法")
N = 3
print(f"   3次までの鏡像の数（直接音を含む）(2N+1)(2N²+2N+3)/3 = {(2 * N + 1) * (2 * N * N + 2 * N + 3) // 3}"
      f"（image_sources で {len(R.image_sources([5, 4, 3], [1, 2, 1.5], 3)[0])}）")
pos, order, hits = R.image_sources([5.0, 4.0, 3.0], [1.0, 2.0, 1.5], 1)
mic = np.array([4.0, 2.0, 1.5])
for p, o in zip(pos, order):
    d = np.linalg.norm(p - mic)
    amp = (0.8 if o else 1.0) * 3.0 / d
    print(f"   鏡像 {p}（次数 {o}）：距離 {d:.3f} m、{d / 343 * 1e3:6.2f} ms、直接音に対する大きさ {20 * np.log10(amp):+.1f} dB")

print("問9 臨界距離と DRR（V = 23.3 m³、T60 = 0.4 s）")
dimq = [3.6, 2.7, 2.4]
rc = R.critical_distance(dimq, 0.4)
print(f"   r_c = {rc:.3f} m（0.057√(V/T) = {0.057 * np.sqrt(np.prod(dimq) / 0.4):.3f}）、1.5 m の DRR {20 * np.log10(rc / 1.5):+.1f} dB、"
      f"カーディオイドなら r_c = {R.critical_distance(dimq, 0.4, 3.0):.3f} m、DRR {20 * np.log10(R.critical_distance(dimq, 0.4, 3.0) / 1.5):+.1f} dB")
print(f"   シュレーダー周波数 {R.schroeder_frequency(0.4, np.prod(dimq)):.0f} Hz"
      f"（導出どおりの係数 2092 なら {np.sqrt(3 * R.C_SOUND ** 3 * 0.4 / (4 * np.pi * 2.2 * np.prod(dimq))):.0f} Hz）")

print("問10 疑似反復")
for icc in (0.9, 0.1):
    print(f"   4 か所 × 5 回、ICC = {icc}：n_eff = 20/(1 + 4·{icc}) = {20 / (1 + 4 * icc):.2f}")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムの中の ok という変数と区別する）
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"\(2N\+1\)\(2N²\+2N\+3\)/3 = (\d+)") == 63, "問8 3次までの鏡像が 63 個")
ok(len(re.findall(r"（次数 1）：距離 5\.000 m、 14\.58 ms", cap.stdout)) == 4, "問8 1次の鏡像のうち4つが同じ 14.58 ms に届く（対称な配置）")

## おわりに

- 自前の `common/room.py` が pyroomacoustics と一致すること、音速・距離減衰、インパルス応答の中身、サビン・アイリングと鏡像法の違い、雑音の床と T60 の不確かさ、ESS と TSP、場所を単位にした集計、部屋のモードと指向性を、数値と音で確かめました。
- このノートブックに入れていないもの：部屋に音を出して測る実機の手順（F の `measure_ir.py --mac`、G の `--write-sweep`・`--analyze`、`t60_stats.py --csv`）と、BME280 で室温を読むこと。Mac のターミナルで、教科書の手順どおりに行ってください。